Section 1 — Environment Setup

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

# Project directory
CURRENT_DIR = Path.cwd()

PROJECT_DIR = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

# Data directories
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
FINAL_DATA_DIR = PROJECT_DIR / "data" / "final"

# Create directories if they don't exist
for folder in [RAW_DATA_DIR, PROCESSED_DATA_DIR, FINAL_DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Random seed for reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Project environment initialized successfully.")
print(f"Raw data location: {RAW_DATA_DIR}")
print(f"Random seed: {RANDOM_SEED}")

Section 2 — Product Master Data

In [ ]:
# Product master data

category_config = {
    "Electronics": {
        "count": 60,
        "subcategories": ["Audio", "Accessories", "Computer Peripherals", "Mobile Accessories"]
    },
    "Home & Kitchen": {
        "count": 70,
        "subcategories": ["Cookware", "Storage", "Home Decor", "Kitchen Appliances"]
    },
    "Clothing": {
        "count": 70,
        "subcategories": ["Men's Wear", "Women's Wear", "Kids' Wear", "Sportswear"]
    },
    "Beauty & Personal Care": {
        "count": 50,
        "subcategories": ["Skincare", "Haircare", "Personal Care", "Grooming"]
    },
    "Sports & Outdoors": {
        "count": 50,
        "subcategories": ["Fitness", "Outdoor Accessories", "Sports Equipment", "Travel Gear"]
    }
}

brand_names = [
    "Averon", "Brightwell", "Cresta", "Dovetail", "Elvora",
    "Finora", "Glint", "Havenix", "Invero", "Jovian",
    "Kavora", "Lumera", "Montra", "Nexora", "Orvella"
]

unit_options = ["Each", "Pack", "Set"]

products = []
product_number = 1

for category, config in category_config.items():
    for i in range(config["count"]):
        subcategory = rng.choice(config["subcategories"])
        brand = rng.choice(brand_names)

        products.append({
            "Product_ID": f"PRD-{product_number:04d}",
            "Product_Name": f"{brand} {subcategory} Product {i + 1:03d}",
            "Category": category,
            "Subcategory": subcategory,
            "Brand": brand,
            "Unit_of_Measure": rng.choice(unit_options)
        })

        product_number += 1

products_df = pd.DataFrame(products)

# Save CSV
products_df.to_csv(RAW_DATA_DIR / "Products.csv", index=False)

print(f"Products generated: {len(products_df):,}")
print("\nProducts by category:")
display(products_df["Category"].value_counts())

print("\nSample records:")
display(products_df.head())

Section 3 — Supplier Master Data

In [ ]:
# Supplier master data

supplier_locations = [
    ("Mumbai", "Maharashtra"),
    ("Pune", "Maharashtra"),
    ("Bengaluru", "Karnataka"),
    ("Mysuru", "Karnataka"),
    ("Chennai", "Tamil Nadu"),
    ("Coimbatore", "Tamil Nadu"),
    ("Hyderabad", "Telangana"),
    ("Delhi", "Delhi"),
    ("Gurugram", "Haryana"),
    ("Noida", "Uttar Pradesh"),
    ("Jaipur", "Rajasthan"),
    ("Ahmedabad", "Gujarat"),
    ("Surat", "Gujarat"),
    ("Kolkata", "West Bengal"),
    ("Indore", "Madhya Pradesh"),
    ("Kochi", "Kerala"),
    ("Nagpur", "Maharashtra"),
    ("Lucknow", "Uttar Pradesh")
]

supplier_types = ["Manufacturer", "Distributor", "Wholesaler"]
supplier_type_weights = [0.30, 0.45, 0.25]

payment_terms_options = [15, 30, 45, 60, 90]

suppliers = []

for i in range(1, 31):
    city, state = supplier_locations[(i - 1) % len(supplier_locations)]

    supplier_type = rng.choice(
        supplier_types,
        p=supplier_type_weights
    )

    suppliers.append({
        "Supplier_ID": f"SUP-{i:03d}",
        "Supplier_Name": f"{city} {supplier_type} Solutions {i:02d}",
        "Supplier_City": city,
        "Supplier_State": state,
        "Supplier_Type": supplier_type,
        "Supplier_Status": "Active",
        "Payment_Terms_Days": int(rng.choice(payment_terms_options))
    })

suppliers_df = pd.DataFrame(suppliers)

# Save CSV
suppliers_df.to_csv(RAW_DATA_DIR / "Suppliers.csv", index=False)

print(f"Suppliers generated: {len(suppliers_df):,}")

print("\nSupplier distribution by type:")
display(suppliers_df["Supplier_Type"].value_counts())

print("\nSupplier distribution by state:")
display(suppliers_df["Supplier_State"].value_counts())

print("\nSample records:")
display(suppliers_df.head(10))

Section 4 — Supplier-Product Mapping

In [ ]:
# Supplier-product mapping

supplier_product_mappings = []
mapping_number = 1

for _, product in products_df.iterrows():

    # Assign 1–6 suppliers per product
    supplier_count = int(
        rng.choice(
            [1, 2, 3, 4, 5, 6],
            p=[0.17, 0.18, 0.22, 0.20, 0.15, 0.08]
        )
    )

    selected_suppliers = rng.choice(
        suppliers_df["Supplier_ID"].values,
        size=supplier_count,
        replace=False
    )

    # Choose one preferred supplier
    preferred_supplier = rng.choice(selected_suppliers)

    # Generate a base product price
    base_price = rng.uniform(150, 12000)

    for supplier_id in selected_suppliers:

        supplier = suppliers_df.loc[
            suppliers_df["Supplier_ID"] == supplier_id
        ].iloc[0]

        # Supplier-specific price variation
        price_factor = rng.uniform(0.88, 1.18)
        unit_price = round(base_price * price_factor, 2)

        # MOQ varies by supplier and product
        moq = int(
            rng.choice(
                [5, 10, 20, 25, 50, 100, 200, 500],
                p=[0.08, 0.12, 0.16, 0.16, 0.20, 0.16, 0.08, 0.04]
            )
        )

        # Supplier-specific lead time
        lead_time = int(rng.integers(5, 36))

        # Use supplier payment terms
        payment_terms = int(supplier["Payment_Terms_Days"])

        supplier_product_mappings.append({
            "Supplier_Product_ID": f"SP-{mapping_number:05d}",
            "Supplier_ID": supplier_id,
            "Product_ID": product["Product_ID"],
            "MOQ_Units": moq,
            "Unit_Purchase_Price": unit_price,
            "Lead_Time_Days": lead_time,
            "Payment_Terms_Days": payment_terms,
            "Is_Preferred_Supplier": (
                supplier_id == preferred_supplier
            )
        })

        mapping_number += 1

supplier_product_df = pd.DataFrame(supplier_product_mappings)

# Save CSV
supplier_product_df.to_csv(
    RAW_DATA_DIR / "Supplier_Product_Mapping.csv",
    index=False
)

# Summary
supplier_count_per_product = (
    supplier_product_df.groupby("Product_ID")["Supplier_ID"]
    .nunique()
)

print(f"Total mappings generated: {len(supplier_product_df):,}")
print(f"Unique products mapped: {supplier_product_df['Product_ID'].nunique():,}")

print("\nNumber of suppliers per product:")
display(supplier_count_per_product.value_counts().sort_index())

print("\nPreferred supplier validation:")
display(
    supplier_product_df.groupby("Product_ID")["Is_Preferred_Supplier"]
    .sum()
    .value_counts()
)

print("\nMOQ distribution:")
display(supplier_product_df["MOQ_Units"].value_counts().sort_index())

print("\nLead time range:")
print(
    f"Minimum: {supplier_product_df['Lead_Time_Days'].min()} days"
)
print(
    f"Maximum: {supplier_product_df['Lead_Time_Days'].max()} days"
)

print("\nSample records:")
display(supplier_product_df.head(10))

Section 5 — Purchase Orders

In [ ]:
# Purchase order header data

from datetime import timedelta

REPORTING_CUTOFF = pd.Timestamp("2026-10-31")
START_DATE = pd.Timestamp("2026-01-01")

number_of_pos = 1500
purchase_orders = []

for i in range(1, number_of_pos + 1):

    # Select supplier
    supplier_id = rng.choice(suppliers_df["Supplier_ID"].values)

    supplier = suppliers_df.loc[
        suppliers_df["Supplier_ID"] == supplier_id
    ].iloc[0]

    # Generate PO date
    po_date = START_DATE + pd.Timedelta(
        days=int(rng.integers(0, 304))
    )

    # Expected delivery date based on supplier lead time
    supplier_lead_time = int(
        rng.integers(5, 36)
    )

    expected_delivery_date = (
        po_date + pd.Timedelta(days=supplier_lead_time)
    )

    # Cancellation probability
    is_cancelled = rng.random() < 0.025

    cancellation_date = pd.NaT
    cancellation_reason = None

    if is_cancelled:
        po_status = "Cancelled"

        cancellation_date = po_date + pd.Timedelta(
            days=int(rng.integers(1, 15))
        )

        cancellation_reason = rng.choice([
            "Business requirement changed",
            "Order placed in error",
            "Supplier unable to fulfil",
            "Budget approval withdrawn",
            "Duplicate purchase order"
        ])

    else:
        # Determine PO status based on expected delivery date
        if expected_delivery_date > REPORTING_CUTOFF:
            po_status = "Open"

        else:
            status_probability = rng.random()

            if status_probability < 0.82:
                po_status = "Closed"
            else:
                po_status = "Overdue"

    # Payment status
    if po_status == "Cancelled":
        payment_status = rng.choice(
            ["Unpaid", "Paid"],
            p=[0.90, 0.10]
        )

    elif po_status == "Closed":
        payment_status = rng.choice(
            ["Paid", "Partially Paid", "Unpaid"],
            p=[0.75, 0.15, 0.10]
        )

    else:
        payment_status = rng.choice(
            ["Paid", "Partially Paid", "Unpaid"],
            p=[0.20, 0.35, 0.45]
        )

    purchase_orders.append({
        "PO_ID": f"PO-{i:05d}",
        "Supplier_ID": supplier_id,
        "PO_Date": po_date,
        "Expected_Delivery_Date": expected_delivery_date,
        "PO_Status": po_status,
        "Payment_Status": payment_status,
        "Cancellation_Date": cancellation_date,
        "Cancellation_Reason": cancellation_reason
    })

purchase_orders_df = pd.DataFrame(purchase_orders)

# Save CSV
purchase_orders_df.to_csv(
    RAW_DATA_DIR / "Purchase_Orders.csv",
    index=False,
    date_format="%Y-%m-%d"
)

# Summary
print(f"Purchase orders generated: {len(purchase_orders_df):,}")

print("\nPO status distribution:")
display(purchase_orders_df["PO_Status"].value_counts())

print("\nPayment status distribution:")
display(purchase_orders_df["Payment_Status"].value_counts())

print("\nCancellation details:")
display(
    purchase_orders_df[
        purchase_orders_df["PO_Status"] == "Cancelled"
    ]["Cancellation_Reason"].value_counts()
)

print("\nDate range:")
print("PO dates:", purchase_orders_df["PO_Date"].min(),
      "to", purchase_orders_df["PO_Date"].max())

print("\nSample records:")
display(purchase_orders_df.head(10))


Section 6 — Purchase Order Lines

In [ ]:
# Purchase order line data

purchase_order_lines = []
line_number = 1

# Generate exactly 2 product lines per PO
for _, po in purchase_orders_df.iterrows():

    supplier_id = po["Supplier_ID"]

    # Find products available from this PO's supplier
    supplier_products = supplier_product_df[
        supplier_product_df["Supplier_ID"] == supplier_id
    ].copy()

    # Prefer the supplier's preferred product relationships,
    # but allow alternative suppliers to be selected too.
    selection_weights = np.where(
        supplier_products["Is_Preferred_Supplier"],
        0.75,
        0.25
    )

    selection_weights = selection_weights / selection_weights.sum()

    # Select 2 different products for this PO
    selected_indices = rng.choice(
        supplier_products.index,
        size=2,
        replace=False,
        p=selection_weights
    )

    selected_products = supplier_products.loc[selected_indices]

    for _, mapping in selected_products.iterrows():

        # Business requirement before applying supplier MOQ
        required_quantity = int(rng.integers(10, 251))

        moq = int(mapping["MOQ_Units"])

        # Generate buyer's intended quantity
        # Some buyers order exactly what is required;
        # others add an additional quantity.
        buyer_order_factor = rng.choice(
            [1.00, 1.10, 1.20, 1.25],
            p=[0.70, 0.15, 0.10, 0.05]
        )

        buyer_intended_quantity = int(
            np.ceil(required_quantity * buyer_order_factor)
        )

        # Apply supplier MOQ
        ordered_quantity = max(
            moq,
            buyer_intended_quantity
        )

        unit_price = float(mapping["Unit_Purchase_Price"])

        line_order_value = round(
            ordered_quantity * unit_price,
            2
        )

        purchase_order_lines.append({
            "PO_Line_ID": f"POL-{line_number:06d}",
            "PO_ID": po["PO_ID"],
            "Product_ID": mapping["Product_ID"],
            "Supplier_ID": supplier_id,
            "Supplier_Product_ID": mapping["Supplier_Product_ID"],
            "Ordered_Quantity": ordered_quantity,
            "Required_Quantity": required_quantity,
            "Unit_Purchase_Price": unit_price,
            "Line_Order_Value": line_order_value
        })

        line_number += 1


purchase_order_lines_df = pd.DataFrame(purchase_order_lines)

# Save CSV
purchase_order_lines_df.to_csv(
    RAW_DATA_DIR / "Purchase_Order_Lines.csv",
    index=False
)

# Summary checks
print(f"Purchase order lines generated: {len(purchase_order_lines_df):,}")
print(f"Unique purchase orders: {purchase_order_lines_df['PO_ID'].nunique():,}")
print(f"Unique products: {purchase_order_lines_df['Product_ID'].nunique():,}")

print("\nLines per purchase order:")
display(
    purchase_order_lines_df.groupby("PO_ID").size().value_counts().sort_index()
)

print("\nQuantity summary:")
display(
    purchase_order_lines_df[
        ["Required_Quantity", "Ordered_Quantity"]
    ].describe().round(2)
)

print("\nMOQ compliance:")
print(
    "Lines ordered below MOQ:",
    (
        purchase_order_lines_df["Ordered_Quantity"]
        < purchase_order_lines_df["Supplier_Product_ID"].map(
            supplier_product_df.set_index("Supplier_Product_ID")["MOQ_Units"]
        )
    ).sum()
)

print("\nTotal purchase order value:")
print(
    f"₹{purchase_order_lines_df['Line_Order_Value'].sum():,.2f}"
)

print("\nSample records:")
display(purchase_order_lines_df.head(10))

Section 7 — Purchase Receipts

In [ ]:
# Purchase receipt data

purchase_receipts = []
receipt_number = 1

for _, line in purchase_order_lines_df.iterrows():

    po = purchase_orders_df.loc[
        purchase_orders_df["PO_ID"] == line["PO_ID"]
    ].iloc[0]

    # Cancelled POs cannot have receipts
    if po["PO_Status"] == "Cancelled":
        continue

    expected_date = po["Expected_Delivery_Date"]
    ordered_qty = int(line["Ordered_Quantity"])

    # Decide the receipt outcome
    if expected_date <= REPORTING_CUTOFF:

        outcome = rng.choice(
            ["Full", "Partial", "None"],
            p=[0.72, 0.18, 0.10]
        )

    elif expected_date <= REPORTING_CUTOFF + pd.Timedelta(days=7):

        # A small possibility of early delivery
        outcome = rng.choice(
            ["Full", "Partial", "None"],
            p=[0.10, 0.10, 0.80]
        )

    else:
        outcome = "None"

    if outcome == "None":
        continue

    # Determine total received quantity
    if outcome == "Full":
        total_received = ordered_qty

    else:
        total_received = int(
            rng.integers(
                max(1, int(ordered_qty * 0.30)),
                max(2, int(ordered_qty * 0.86))
            )
        )

    # Rejected quantities occur only on partial deliveries
    total_rejected = 0

    if outcome == "Partial" and rng.random() < 0.20:
        total_rejected = int(
            rng.integers(
                1,
                max(2, int(total_received * 0.05) + 1)
            )
        )

    total_accepted = total_received - total_rejected

    # Number of delivery records for this PO line
    number_of_receipts = int(
        rng.integers(1, min(3, total_received) + 1)
    )

    # Split total received quantity across deliveries
    split_points = sorted(
        rng.choice(
            np.arange(1, total_received),
            size=number_of_receipts - 1,
            replace=False
        )
    ) if number_of_receipts > 1 else []

    receipt_quantities = np.diff(
        [0] + list(split_points) + [total_received]
    )

    # Generate delivery dates
    if expected_date <= REPORTING_CUTOFF:
        first_delivery_date = expected_date + pd.Timedelta(
            days=int(rng.integers(-3, 15))
        )
    else:
        first_delivery_date = expected_date - pd.Timedelta(
            days=int(rng.integers(1, 8))
        )

    first_delivery_date = max(
        po["PO_Date"],
        min(first_delivery_date, REPORTING_CUTOFF)
    )

    delivery_dates = [first_delivery_date]

    for _ in range(1, number_of_receipts):
        next_date = delivery_dates[-1] + pd.Timedelta(
            days=int(rng.integers(1, 8))
        )

        delivery_dates.append(
            min(next_date, REPORTING_CUTOFF)
        )

    # Split rejected quantity across receipt records
    rejected_quantities = np.zeros(
        number_of_receipts,
        dtype=int
    )

    if total_rejected > 0:
        rejected_quantities[
            -1
        ] = total_rejected

    # Create receipt records
    for j in range(number_of_receipts):

        received_qty = int(receipt_quantities[j])
        rejected_qty = int(rejected_quantities[j])
        accepted_qty = received_qty - rejected_qty

        receipt_status = (
            "Partially Rejected"
            if rejected_qty > 0
            else "Accepted"
        )

        purchase_receipts.append({
            "Receipt_ID": f"REC-{receipt_number:06d}",
            "PO_Line_ID": line["PO_Line_ID"],
            "Receipt_Date": delivery_dates[j],
            "Received_Quantity": received_qty,
            "Accepted_Quantity": accepted_qty,
            "Rejected_Quantity": rejected_qty,
            "Receipt_Status": receipt_status
        })

        receipt_number += 1


purchase_receipts_df = pd.DataFrame(purchase_receipts)

# Save CSV
purchase_receipts_df.to_csv(
    RAW_DATA_DIR / "Purchase_Receipts.csv",
    index=False,
    date_format="%Y-%m-%d"
)

# Summary checks
print(f"Receipt records generated: {len(purchase_receipts_df):,}")
print(
    "PO lines with receipts:",
    purchase_receipts_df["PO_Line_ID"].nunique()
)

print("\nReceipt status distribution:")
display(purchase_receipts_df["Receipt_Status"].value_counts())

print("\nReceipt quantity summary:")
display(
    purchase_receipts_df[
        ["Received_Quantity", "Accepted_Quantity", "Rejected_Quantity"]
    ].sum().to_frame("Total_Units")
)

print("\nReceipt date range:")
print(
    purchase_receipts_df["Receipt_Date"].min(),
    "to",
    purchase_receipts_df["Receipt_Date"].max()
)

# Validate accepted + rejected = received
quantity_check = (
    purchase_receipts_df["Accepted_Quantity"]
    + purchase_receipts_df["Rejected_Quantity"]
    == purchase_receipts_df["Received_Quantity"]
)

print("\nAccepted + rejected = received:")
print("All records valid:", quantity_check.all())

# Validate received quantity does not exceed ordered quantity
received_by_line = (
    purchase_receipts_df.groupby("PO_Line_ID")["Received_Quantity"]
    .sum()
)

ordered_by_line = purchase_order_lines_df.set_index(
    "PO_Line_ID"
)["Ordered_Quantity"]

over_received_lines = (
    received_by_line
    > ordered_by_line.loc[received_by_line.index]
).sum()

print("\nPO lines with over-delivery:", over_received_lines)

print("\nSample records:")
display(purchase_receipts_df.head(10))

In [ ]:
# Reconcile PO statuses using actual accepted receipts

# Preserve the original status for reference
purchase_orders_df["PO_Status_Provisional"] = purchase_orders_df["PO_Status"]

# Convert dates to datetime
purchase_orders_df["Expected_Delivery_Date"] = pd.to_datetime(
    purchase_orders_df["Expected_Delivery_Date"]
)

purchase_receipts_df["Receipt_Date"] = pd.to_datetime(
    purchase_receipts_df["Receipt_Date"]
)

# Total accepted quantity for each PO line
accepted_by_line = (
    purchase_receipts_df
    .groupby("PO_Line_ID", as_index=False)["Accepted_Quantity"]
    .sum()
)

# Bring accepted quantities onto all PO lines
line_status_df = purchase_order_lines_df[
    ["PO_ID", "PO_Line_ID", "Ordered_Quantity"]
].merge(
    accepted_by_line,
    on="PO_Line_ID",
    how="left"
)

line_status_df["Accepted_Quantity"] = (
    line_status_df["Accepted_Quantity"].fillna(0)
)

# Check whether each line is fully accepted
line_status_df["Line_Fully_Accepted"] = (
    line_status_df["Accepted_Quantity"]
    >= line_status_df["Ordered_Quantity"]
)

# Determine whether every line on each PO is fully accepted
po_fulfilment_df = (
    line_status_df
    .groupby("PO_ID", as_index=False)
    .agg(
        Total_PO_Lines=("PO_Line_ID", "count"),
        Fully_Accepted_Lines=("Line_Fully_Accepted", "sum")
    )
)

po_fulfilment_df["All_Lines_Fully_Accepted"] = (
    po_fulfilment_df["Total_PO_Lines"]
    == po_fulfilment_df["Fully_Accepted_Lines"]
)

# Merge fulfilment results into PO headers
purchase_orders_df = purchase_orders_df.merge(
    po_fulfilment_df[
        ["PO_ID", "All_Lines_Fully_Accepted"]
    ],
    on="PO_ID",
    how="left"
)

# Assign final PO status
purchase_orders_df["PO_Status"] = np.select(
    [
        purchase_orders_df["PO_Status_Provisional"] == "Cancelled",
        purchase_orders_df["All_Lines_Fully_Accepted"],
        purchase_orders_df["Expected_Delivery_Date"] <= REPORTING_CUTOFF
    ],
    [
        "Cancelled",
        "Closed",
        "Overdue"
    ],
    default="Open"
)

# Save the corrected reconciled PO data
purchase_orders_df.to_csv(
    PROCESSED_DATA_DIR / "Purchase_Orders_Reconciled.csv",
    index=False,
    date_format="%Y-%m-%d"
)

display(
    purchase_orders_df["PO_Status"]
    .value_counts()
    .to_frame("PO_Count")
)

# Summary
print("Final PO status distribution:")
display(
    purchase_orders_df["PO_Status"].value_counts().to_frame("PO_Count")
)

print("\nProvisional vs final status:")
display(
    pd.crosstab(
        purchase_orders_df["PO_Status_Provisional"],
        purchase_orders_df["PO_Status"],
        margins=True
    )
)

print("\nValidation:")
print(
    "Cancelled POs:",
    (purchase_orders_df["PO_Status"] == "Cancelled").sum()
)

print(
    "POs without a fulfilment result:",
    purchase_orders_df["All_Lines_Fully_Accepted"].isna().sum()
)

In [ ]:
# Basic relationship validation across all six datasets

# 1. Products referenced in supplier mapping
invalid_mapping_products = (
    ~supplier_product_df["Product_ID"].isin(
        products_df["Product_ID"]
    )
).sum()

# 2. Suppliers referenced in supplier mapping
invalid_mapping_suppliers = (
    ~supplier_product_df["Supplier_ID"].isin(
        suppliers_df["Supplier_ID"]
    )
).sum()

# 3. PO suppliers must exist in supplier master
invalid_po_suppliers = (
    ~purchase_orders_df["Supplier_ID"].isin(
        suppliers_df["Supplier_ID"]
    )
).sum()

# 4. PO lines must reference valid POs
invalid_line_pos = (
    ~purchase_order_lines_df["PO_ID"].isin(
        purchase_orders_df["PO_ID"]
    )
).sum()

# 5. PO lines must reference valid products
invalid_line_products = (
    ~purchase_order_lines_df["Product_ID"].isin(
        products_df["Product_ID"]
    )
).sum()

# 6. PO lines must reference valid supplier-product mappings
invalid_line_mappings = (
    ~purchase_order_lines_df["Supplier_Product_ID"].isin(
        supplier_product_df["Supplier_Product_ID"]
    )
).sum()

# 7. Supplier on PO line must match PO header supplier
po_supplier_check = purchase_order_lines_df.merge(
    purchase_orders_df[["PO_ID", "Supplier_ID"]],
    on="PO_ID",
    suffixes=("_Line", "_Header")
)

supplier_mismatches = (
    po_supplier_check["Supplier_ID_Line"]
    != po_supplier_check["Supplier_ID_Header"]
).sum()

# 8. Receipt records must reference valid PO lines
invalid_receipt_lines = (
    ~purchase_receipts_df["PO_Line_ID"].isin(
        purchase_order_lines_df["PO_Line_ID"]
    )
).sum()

# 9. Cancelled POs must not have receipts
cancelled_po_ids = set(
    purchase_orders_df.loc[
        purchase_orders_df["PO_Status"] == "Cancelled",
        "PO_ID"
    ]
)

cancelled_line_ids = set(
    purchase_order_lines_df.loc[
        purchase_order_lines_df["PO_ID"].isin(cancelled_po_ids),
        "PO_Line_ID"
    ]
)

cancelled_receipts = (
    purchase_receipts_df["PO_Line_ID"]
    .isin(cancelled_line_ids)
).sum()

# Display results
validation_results = pd.DataFrame({
    "Validation": [
        "Invalid products in supplier mapping",
        "Invalid suppliers in supplier mapping",
        "Invalid suppliers in PO headers",
        "Invalid POs in PO lines",
        "Invalid products in PO lines",
        "Invalid supplier mappings in PO lines",
        "PO header and line supplier mismatches",
        "Invalid PO lines in receipts",
        "Receipts against cancelled POs"
    ],
    "Issue_Count": [
        invalid_mapping_products,
        invalid_mapping_suppliers,
        invalid_po_suppliers,
        invalid_line_pos,
        invalid_line_products,
        invalid_line_mappings,
        supplier_mismatches,
        invalid_receipt_lines,
        cancelled_receipts
    ]
})

display(validation_results)

print(
    "\nAll relationship checks passed:",
    validation_results["Issue_Count"].sum() == 0
)

In [ ]:
%whos DataFrame

## Step 4 — Load Data into DuckDB

In [ ]:
import duckdb

# Database path
DB_PATH = PROJECT_DIR / "procurewise_procurement.duckdb"

# Connect to DuckDB
con = duckdb.connect(str(DB_PATH))

print("DuckDB connection successful.")
print("Database:", DB_PATH)

In [ ]:
# Load datasets into DuckDB

datasets = {
    "Products": products_df,
    "Suppliers": suppliers_df,
    "Supplier_Product_Mapping": supplier_product_df,
    "Purchase_Orders": purchase_orders_df,
    "Purchase_Order_Lines": purchase_order_lines_df,
    "Purchase_Receipts": purchase_receipts_df
}

for table_name, df in datasets.items():
    con.register("temp_df", df)

    con.execute(
        f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM temp_df"
    )

    con.unregister("temp_df")

    print(f"{table_name}: {len(df):,} rows loaded")

# Replace PO table with reconciled statuses
con.register("temp_df", purchase_orders_df)

con.execute("""
    CREATE OR REPLACE TABLE Purchase_Orders AS
    SELECT * FROM temp_df
""")

con.unregister("temp_df")

print("\nAll six tables loaded successfully.")

In [ ]:
# Verify DuckDB tables and row counts

table_check = con.execute("""
    SELECT
        table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

display(table_check)

print("\nRow counts:")

for table_name in table_check["table_name"]:
    row_count = con.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    print(f"{table_name}: {row_count:,}")

In [ ]:
po_summary = con.execute("""
    SELECT
        PO_Status,
        COUNT(*) AS PO_Count,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS PO_Percentage
    FROM Purchase_Orders
    GROUP BY PO_Status
    ORDER BY PO_Count DESC
""").df()

display(po_summary)

In [ ]:
po_monthly_summary = con.execute("""
    SELECT
        DATE_TRUNC('month', PO_Date)::DATE AS PO_Month,
        COUNT(*) AS Total_POs,
        COUNT(DISTINCT Supplier_ID) AS Active_Suppliers,
        SUM(
            CASE WHEN PO_Status = 'Cancelled'
                 THEN 1 ELSE 0 END
        ) AS Cancelled_POs
    FROM Purchase_Orders
    GROUP BY 1
    ORDER BY 1
""").df()

display(po_monthly_summary)

In [ ]:
monthly_purchase_value = con.execute("""
    SELECT
        DATE_TRUNC('month', po.PO_Date)::DATE AS PO_Month,
        COUNT(DISTINCT po.PO_ID) AS Total_POs,
        COUNT(pol.PO_Line_ID) AS Total_PO_Lines,
        ROUND(SUM(pol.Line_Order_Value), 2) AS Purchase_Order_Value,
        ROUND(
            AVG(pol.Line_Order_Value), 2
        ) AS Average_Line_Value
    FROM Purchase_Orders po
    JOIN Purchase_Order_Lines pol
        ON po.PO_ID = pol.PO_ID
    WHERE po.PO_Status != 'Cancelled'
    GROUP BY 1
    ORDER BY 1
""").df()

display(monthly_purchase_value)


In [ ]:
category_purchase_summary = con.execute("""
    SELECT
        p.Category,
        COUNT(DISTINCT pol.PO_ID) AS Total_POs,
        COUNT(DISTINCT pol.Product_ID) AS Products_Ordered,
        SUM(pol.Ordered_Quantity) AS Total_Ordered_Units,
        ROUND(SUM(pol.Line_Order_Value), 2) AS Purchase_Value,
        ROUND(
            SUM(pol.Line_Order_Value) * 100.0
            / SUM(SUM(pol.Line_Order_Value)) OVER (),
            2
        ) AS Purchase_Value_Percentage
    FROM Purchase_Order_Lines pol
    JOIN Purchase_Orders po
        ON pol.PO_ID = po.PO_ID
    JOIN Products p
        ON pol.Product_ID = p.Product_ID
    WHERE po.PO_Status != 'Cancelled'
    GROUP BY p.Category
    ORDER BY Purchase_Value DESC
""").df()

display(category_purchase_summary)

In [ ]:
supplier_purchase_summary = con.execute("""
    SELECT
        s.Supplier_ID,
        s.Supplier_Name,
        s.Supplier_City,
        s.Supplier_Type,
        COUNT(DISTINCT po.PO_ID) AS Total_POs,
        COUNT(pol.PO_Line_ID) AS Total_PO_Lines,
        ROUND(SUM(pol.Line_Order_Value), 2) AS Purchase_Value,
        SUM(pol.Ordered_Quantity) AS Units_Ordered
    FROM Purchase_Orders po
    JOIN Purchase_Order_Lines pol
        ON po.PO_ID = pol.PO_ID
    JOIN Suppliers s
        ON po.Supplier_ID = s.Supplier_ID
    WHERE po.PO_Status != 'Cancelled'
    GROUP BY
        s.Supplier_ID,
        s.Supplier_Name,
        s.Supplier_City,
        s.Supplier_Type
    ORDER BY Purchase_Value DESC
""").df()

display(supplier_purchase_summary)

In [ ]:
supplier_concentration = con.execute("""
    WITH supplier_spend AS (
        SELECT
            po.Supplier_ID,
            ROUND(SUM(pol.Line_Order_Value), 2) AS Purchase_Value
        FROM Purchase_Orders po
        JOIN Purchase_Order_Lines pol
            ON po.PO_ID = pol.PO_ID
        WHERE po.PO_Status != 'Cancelled'
        GROUP BY po.Supplier_ID
    ),
    ranked_suppliers AS (
        SELECT
            Supplier_ID,
            Purchase_Value,
            ROUND(
                Purchase_Value * 100.0
                / SUM(Purchase_Value) OVER (),
                2
            ) AS Purchase_Share_Pct,
            SUM(Purchase_Value) OVER (
                ORDER BY Purchase_Value DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) * 100.0
            / SUM(Purchase_Value) OVER () AS Cumulative_Share_Pct
        FROM supplier_spend
    )
    SELECT
        Supplier_ID,
        Purchase_Value,
        Purchase_Share_Pct,
        ROUND(Cumulative_Share_Pct, 2) AS Cumulative_Share_Pct
    FROM ranked_suppliers
    ORDER BY Purchase_Value DESC
""").df()

display(supplier_concentration)

In [ ]:
product_sourcing_summary = con.execute("""
    SELECT
        Supplier_Count,
        COUNT(*) AS Product_Count,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS Product_Percentage
    FROM (
        SELECT
            Product_ID,
            COUNT(DISTINCT Supplier_ID) AS Supplier_Count
        FROM Supplier_Product_Mapping
        GROUP BY Product_ID
    ) product_suppliers
    GROUP BY Supplier_Count
    ORDER BY Supplier_Count
""").df()

display(product_sourcing_summary)

In [ ]:
preferred_moq_summary = con.execute("""
    SELECT
        p.Category,
        COUNT(*) AS Product_Count,
        ROUND(AVG(spm.MOQ_Units), 2) AS Average_Preferred_MOQ,
        MEDIAN(spm.MOQ_Units) AS Median_Preferred_MOQ,
        MAX(spm.MOQ_Units) AS Maximum_Preferred_MOQ,
        SUM(
            CASE WHEN spm.MOQ_Units >= 100
                 THEN 1 ELSE 0 END
        ) AS Products_MOQ_100_Or_More
    FROM Supplier_Product_Mapping spm
    JOIN Products p
        ON spm.Product_ID = p.Product_ID
    WHERE spm.Is_Preferred_Supplier = TRUE
    GROUP BY p.Category
    ORDER BY Average_Preferred_MOQ DESC
""").df()

display(preferred_moq_summary)

In [ ]:
supplier_option_summary = con.execute("""
    WITH supplier_options AS (
        SELECT
            Product_ID,
            COUNT(*) AS Supplier_Count,
            MIN(MOQ_Units) AS Lowest_MOQ,
            MIN(Unit_Purchase_Price) AS Lowest_Unit_Price,
            MIN(Lead_Time_Days) AS Shortest_Lead_Time,
            MAX(MOQ_Units) AS Highest_MOQ
        FROM Supplier_Product_Mapping
        GROUP BY Product_ID
    ),
    preferred_supplier AS (
        SELECT
            Product_ID,
            Supplier_ID AS Preferred_Supplier_ID,
            MOQ_Units AS Preferred_MOQ,
            Unit_Purchase_Price AS Preferred_Unit_Price,
            Lead_Time_Days AS Preferred_Lead_Time
        FROM Supplier_Product_Mapping
        WHERE Is_Preferred_Supplier = TRUE
    )
    SELECT
        p.Category,
        COUNT(*) AS Product_Count,
        SUM(CASE WHEN so.Lowest_MOQ < ps.Preferred_MOQ
                 THEN 1 ELSE 0 END) AS Products_With_Lower_Alternative_MOQ,
        SUM(CASE WHEN so.Lowest_Unit_Price < ps.Preferred_Unit_Price
                 THEN 1 ELSE 0 END) AS Products_With_Cheaper_Alternative,
        SUM(CASE WHEN so.Shortest_Lead_Time < ps.Preferred_Lead_Time
                 THEN 1 ELSE 0 END) AS Products_With_Faster_Alternative
    FROM Products p
    JOIN supplier_options so
        ON p.Product_ID = so.Product_ID
    JOIN preferred_supplier ps
        ON p.Product_ID = ps.Product_ID
    GROUP BY p.Category
    ORDER BY p.Category
""").df()

display(supplier_option_summary)

In [ ]:
moq_distribution = con.execute("""
    SELECT
        MOQ_Units,
        COUNT(*) AS Supplier_Product_Records,
        COUNT(DISTINCT Product_ID) AS Unique_Products,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS Record_Percentage
    FROM Supplier_Product_Mapping
    GROUP BY MOQ_Units
    ORDER BY MOQ_Units
""").df()

display(moq_distribution)

In [ ]:
moq_impact_summary = con.execute("""
    SELECT
        COUNT(*) AS Total_Order_Lines,

        SUM(Required_Quantity) AS Total_Required_Units,
        SUM(Ordered_Quantity) AS Total_Ordered_Units,

        SUM(
            GREATEST(
                LEAST(Ordered_Quantity, MOQ_Units)
                - Required_Quantity,
                0
            )
        ) AS MOQ_Driven_Excess_Units,

        SUM(
            GREATEST(
                Ordered_Quantity - GREATEST(Required_Quantity, MOQ_Units),
                0
            )
        ) AS Buyer_Excess_Above_MOQ_Units,

        SUM(
            GREATEST(Ordered_Quantity - Required_Quantity, 0)
        ) AS Total_Excess_Units

    FROM Purchase_Order_Lines pol
    JOIN Supplier_Product_Mapping spm
        ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
    JOIN Purchase_Orders po
        ON pol.PO_ID = po.PO_ID
    WHERE po.PO_Status != 'Cancelled'
""").df()

display(moq_impact_summary)

In [ ]:
moq_excess_value = con.execute("""
    WITH line_excess AS (
        SELECT
            pol.PO_Line_ID,
            pol.Required_Quantity,
            pol.Ordered_Quantity,
            pol.Unit_Purchase_Price,
            spm.MOQ_Units,

            GREATEST(
                LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                - pol.Required_Quantity,
                0
            ) AS MOQ_Driven_Excess_Units,

            GREATEST(
                pol.Ordered_Quantity
                - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                0
            ) AS Buyer_Excess_Units

        FROM Purchase_Order_Lines pol
        JOIN Supplier_Product_Mapping spm
            ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        WHERE po.PO_Status != 'Cancelled'
    )

    SELECT
        COUNT(*) AS Total_Order_Lines,

        ROUND(
            SUM(MOQ_Driven_Excess_Units * Unit_Purchase_Price),
            2
        ) AS MOQ_Driven_Excess_Value,

        ROUND(
            SUM(Buyer_Excess_Units * Unit_Purchase_Price),
            2
        ) AS Buyer_Excess_Value,

        ROUND(
            SUM(
                (MOQ_Driven_Excess_Units + Buyer_Excess_Units)
                * Unit_Purchase_Price
            ),
            2
        ) AS Total_Excess_Purchase_Value

    FROM line_excess
""").df()

display(moq_excess_value)

In [ ]:
moq_excess_value = con.execute("""
    WITH line_over_moq AS (
        SELECT
            pol.PO_Line_ID,
            pol.Required_Quantity,
            pol.Ordered_Quantity,
            pol.Unit_Purchase_Price,
            spm.MOQ_Units,

            GREATEST(
                LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                - pol.Required_Quantity,
                0
            ) AS MOQ_Driven_Over_MOQ_Units,

            GREATEST(
                pol.Ordered_Quantity
                - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                0
            ) AS Buyer_Driven_Over_MOQ_Units

        FROM Purchase_Order_Lines pol
        JOIN Supplier_Product_Mapping spm
            ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        WHERE po.PO_Status != 'Cancelled'
    )

    SELECT
        COUNT(*) AS Total_Order_Lines,

        ROUND(
            SUM(MOQ_Driven_Over_MOQ_Units * Unit_Purchase_Price),
            2
        ) AS MOQ_Driven_Over_MOQ_Value,

        ROUND(
            SUM(Buyer_Driven_Over_MOQ_Units * Unit_Purchase_Price),
            2
        ) AS Buyer_Driven_Over_MOQ_Value,

        ROUND(
            SUM(
                (MOQ_Driven_Over_MOQ_Units
                + Buyer_Driven_Over_MOQ_Units)
                * Unit_Purchase_Price
            ),
            2
        ) AS Total_Over_MOQ_Purchase_Value

    FROM line_over_moq
""").df()

display(moq_excess_value)

In [ ]:
over_moq_category_summary = con.execute("""
    WITH line_over_moq AS (
        SELECT
            pol.PO_Line_ID,
            pol.Product_ID,
            pol.Required_Quantity,
            pol.Ordered_Quantity,
            pol.Unit_Purchase_Price,
            spm.MOQ_Units,

            GREATEST(
                LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                - pol.Required_Quantity,
                0
            ) AS MOQ_Driven_Over_MOQ_Units,

            GREATEST(
                pol.Ordered_Quantity
                - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                0
            ) AS Buyer_Driven_Over_MOQ_Units

        FROM Purchase_Order_Lines pol
        JOIN Supplier_Product_Mapping spm
            ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        WHERE po.PO_Status != 'Cancelled'
    )

    SELECT
        p.Category,
        COUNT(*) AS Order_Lines,

        SUM(l.MOQ_Driven_Over_MOQ_Units)
            AS MOQ_Driven_Over_MOQ_Units,

        SUM(l.Buyer_Driven_Over_MOQ_Units)
            AS Buyer_Driven_Over_MOQ_Units,

        SUM(
            (l.MOQ_Driven_Over_MOQ_Units
             + l.Buyer_Driven_Over_MOQ_Units)
            * l.Unit_Purchase_Price
        ) AS Total_Over_MOQ_Purchase_Value

    FROM line_over_moq l
    JOIN Products p
        ON l.Product_ID = p.Product_ID
    GROUP BY p.Category
    ORDER BY Total_Over_MOQ_Purchase_Value DESC
""").df()

display(over_moq_category_summary)

In [ ]:
over_moq_supplier_summary = con.execute("""
    WITH line_over_moq AS (
        SELECT
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Required_Quantity,
            pol.Ordered_Quantity,
            pol.Unit_Purchase_Price,
            spm.MOQ_Units,

            GREATEST(
                LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                - pol.Required_Quantity,
                0
            ) AS MOQ_Driven_Over_MOQ_Units,

            GREATEST(
                pol.Ordered_Quantity
                - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                0
            ) AS Buyer_Driven_Over_MOQ_Units

        FROM Purchase_Order_Lines pol
        JOIN Supplier_Product_Mapping spm
            ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        WHERE po.PO_Status != 'Cancelled'
    )

    SELECT
        s.Supplier_ID,
        s.Supplier_Name,
        COUNT(*) AS Order_Lines,

        SUM(l.MOQ_Driven_Over_MOQ_Units)
            AS MOQ_Driven_Over_MOQ_Units,

        SUM(l.Buyer_Driven_Over_MOQ_Units)
            AS Buyer_Driven_Over_MOQ_Units,

        ROUND(
            SUM(
                (l.MOQ_Driven_Over_MOQ_Units
                 + l.Buyer_Driven_Over_MOQ_Units)
                * l.Unit_Purchase_Price
            ),
            2
        ) AS Total_Over_MOQ_Purchase_Value

    FROM line_over_moq l
    JOIN Suppliers s
        ON l.Supplier_ID = s.Supplier_ID
    GROUP BY s.Supplier_ID, s.Supplier_Name
    ORDER BY Total_Over_MOQ_Purchase_Value DESC
""").df()

display(over_moq_supplier_summary)

In [ ]:
over_moq_supplier_type = con.execute("""
    WITH line_over_moq AS (
        SELECT
            pol.Supplier_ID,
            pol.Required_Quantity,
            pol.Ordered_Quantity,
            pol.Unit_Purchase_Price,
            spm.MOQ_Units,

            GREATEST(
                LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                - pol.Required_Quantity,
                0
            ) AS MOQ_Driven_Over_MOQ_Units,

            GREATEST(
                pol.Ordered_Quantity
                - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                0
            ) AS Buyer_Driven_Over_MOQ_Units

        FROM Purchase_Order_Lines pol
        JOIN Supplier_Product_Mapping spm
            ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        WHERE po.PO_Status != 'Cancelled'
    )

    SELECT
        s.Supplier_Type,
        COUNT(*) AS Order_Lines,
        COUNT(DISTINCT l.Supplier_ID) AS Supplier_Count,

        SUM(l.MOQ_Driven_Over_MOQ_Units)
            AS MOQ_Driven_Over_MOQ_Units,

        SUM(l.Buyer_Driven_Over_MOQ_Units)
            AS Buyer_Driven_Over_MOQ_Units,

        ROUND(
            SUM(
                (l.MOQ_Driven_Over_MOQ_Units
                 + l.Buyer_Driven_Over_MOQ_Units)
                * l.Unit_Purchase_Price
            ),
            2
        ) AS Total_Over_MOQ_Purchase_Value

    FROM line_over_moq l
    JOIN Suppliers s
        ON l.Supplier_ID = s.Supplier_ID
    GROUP BY s.Supplier_Type
    ORDER BY Total_Over_MOQ_Purchase_Value DESC
""").df()

display(over_moq_supplier_type)

In [ ]:
over_moq_product_summary = con.execute("""
    WITH line_over_moq AS (
        SELECT
            pol.Product_ID,
            pol.Required_Quantity,
            pol.Ordered_Quantity,
            pol.Unit_Purchase_Price,
            spm.MOQ_Units,

            GREATEST(
                LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                - pol.Required_Quantity,
                0
            ) AS MOQ_Driven_Over_MOQ_Units,

            GREATEST(
                pol.Ordered_Quantity
                - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                0
            ) AS Buyer_Driven_Over_MOQ_Units

        FROM Purchase_Order_Lines pol
        JOIN Supplier_Product_Mapping spm
            ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        WHERE po.PO_Status != 'Cancelled'
    )

    SELECT
        p.Product_ID,
        p.Product_Name,
        p.Category,
        COUNT(*) AS Order_Lines,

        SUM(l.Required_Quantity) AS Required_Units,
        SUM(l.Ordered_Quantity) AS Ordered_Units,

        SUM(l.MOQ_Driven_Over_MOQ_Units)
            AS MOQ_Driven_Over_MOQ_Units,

        SUM(l.Buyer_Driven_Over_MOQ_Units)
            AS Buyer_Driven_Over_MOQ_Units,

        ROUND(
            SUM(
                (l.MOQ_Driven_Over_MOQ_Units
                 + l.Buyer_Driven_Over_MOQ_Units)
                * l.Unit_Purchase_Price
            ),
            2
        ) AS Total_Over_MOQ_Purchase_Value

    FROM line_over_moq l
    JOIN Products p
        ON l.Product_ID = p.Product_ID
    GROUP BY p.Product_ID, p.Product_Name, p.Category
    HAVING SUM(
        l.MOQ_Driven_Over_MOQ_Units
        + l.Buyer_Driven_Over_MOQ_Units
    ) > 0
    ORDER BY Total_Over_MOQ_Purchase_Value DESC
""").df()

display(over_moq_product_summary.head(20))

In [ ]:
top_product_supplier_options = con.execute("""
    WITH top_products AS (
        SELECT Product_ID
        FROM (
            SELECT
                pol.Product_ID,
                SUM(
                    (
                        GREATEST(
                            LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                            - pol.Required_Quantity,
                            0
                        )
                        +
                        GREATEST(
                            pol.Ordered_Quantity
                            - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                            0
                        )
                    ) * pol.Unit_Purchase_Price
                ) AS Over_MOQ_Value
            FROM Purchase_Order_Lines pol
            JOIN Supplier_Product_Mapping spm
                ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
            JOIN Purchase_Orders po
                ON pol.PO_ID = po.PO_ID
            WHERE po.PO_Status != 'Cancelled'
            GROUP BY pol.Product_ID
            ORDER BY Over_MOQ_Value DESC
            LIMIT 10
        )
    )

    SELECT
        p.Product_ID,
        p.Product_Name,
        s.Supplier_ID,
        s.Supplier_Name,
        spm.Is_Preferred_Supplier,
        spm.MOQ_Units,
        spm.Unit_Purchase_Price,
        spm.Lead_Time_Days
    FROM top_products tp
    JOIN Products p
        ON tp.Product_ID = p.Product_ID
    JOIN Supplier_Product_Mapping spm
        ON p.Product_ID = spm.Product_ID
    JOIN Suppliers s
        ON spm.Supplier_ID = s.Supplier_ID
    ORDER BY
        p.Product_ID,
        spm.Is_Preferred_Supplier DESC,
        spm.MOQ_Units,
        spm.Unit_Purchase_Price
""").df()

display(top_product_supplier_options)

In [ ]:
top_product_supplier_impact = con.execute("""
    WITH line_over_moq AS (
        SELECT
            pol.Product_ID,
            pol.Supplier_ID,
            COUNT(*) AS Order_Lines,

            SUM(
                GREATEST(
                    LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                    - pol.Required_Quantity,
                    0
                )
            ) AS MOQ_Driven_Over_MOQ_Units,

            SUM(
                GREATEST(
                    pol.Ordered_Quantity
                    - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                    0
                )
            ) AS Buyer_Driven_Over_MOQ_Units,

            SUM(
                (
                    GREATEST(
                        LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                        - pol.Required_Quantity,
                        0
                    )
                    +
                    GREATEST(
                        pol.Ordered_Quantity
                        - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                        0
                    )
                ) * pol.Unit_Purchase_Price
            ) AS Total_Over_MOQ_Purchase_Value

        FROM Purchase_Order_Lines pol
        JOIN Supplier_Product_Mapping spm
            ON pol.Supplier_Product_ID = spm.Supplier_Product_ID
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        WHERE po.PO_Status != 'Cancelled'
        GROUP BY pol.Product_ID, pol.Supplier_ID
    ),
    top_products AS (
        SELECT Product_ID
        FROM (
            SELECT
                Product_ID,
                SUM(Total_Over_MOQ_Purchase_Value) AS Over_MOQ_Value
            FROM line_over_moq
            GROUP BY Product_ID
            ORDER BY Over_MOQ_Value DESC
            LIMIT 10
        )
    )

    SELECT
        p.Product_ID,
        p.Product_Name,
        s.Supplier_ID,
        s.Supplier_Name,
        spm.Is_Preferred_Supplier,
        spm.MOQ_Units,
        l.Order_Lines,
        l.MOQ_Driven_Over_MOQ_Units,
        l.Buyer_Driven_Over_MOQ_Units,
        ROUND(l.Total_Over_MOQ_Purchase_Value, 2)
            AS Total_Over_MOQ_Purchase_Value

    FROM line_over_moq l
    JOIN top_products tp
        ON l.Product_ID = tp.Product_ID
    JOIN Products p
        ON l.Product_ID = p.Product_ID
    JOIN Suppliers s
        ON l.Supplier_ID = s.Supplier_ID
    JOIN Supplier_Product_Mapping spm
        ON l.Product_ID = spm.Product_ID
        AND l.Supplier_ID = spm.Supplier_ID

    ORDER BY
        p.Product_ID,
        l.Total_Over_MOQ_Purchase_Value DESC
""").df()

display(top_product_supplier_impact)

In [ ]:
delivery_performance_summary = con.execute("""
    WITH line_receipts AS (
        SELECT
            pol.PO_Line_ID,
            pol.PO_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date,

            COALESCE(SUM(pr.Accepted_Quantity), 0)
                AS Total_Accepted_Quantity,

            MAX(
                CASE
                    WHEN pr.Accepted_Quantity > 0
                    THEN pr.Receipt_Date
                END
            ) AS Last_Accepted_Receipt_Date

        FROM Purchase_Order_Lines pol
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        LEFT JOIN Purchase_Receipts pr
            ON pol.PO_Line_ID = pr.PO_Line_ID
        WHERE po.PO_Status = 'Closed'
        GROUP BY
            pol.PO_Line_ID,
            pol.PO_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date
    )

    SELECT
        COUNT(*) AS Completed_PO_Lines,

        SUM(
            CASE
                WHEN Total_Accepted_Quantity >= Ordered_Quantity
                THEN 1 ELSE 0
            END
        ) AS Fully_Accepted_Lines,

        SUM(
            CASE
                WHEN Total_Accepted_Quantity >= Ordered_Quantity
                 AND Last_Accepted_Receipt_Date
                     <= Expected_Delivery_Date
                THEN 1 ELSE 0
            END
        ) AS On_Time_Lines,

        SUM(
            CASE
                WHEN Total_Accepted_Quantity >= Ordered_Quantity
                 AND Last_Accepted_Receipt_Date
                     > Expected_Delivery_Date
                THEN 1 ELSE 0
            END
        ) AS Late_Lines

    FROM line_receipts
""").df()

display(delivery_performance_summary)

In [ ]:
supplier_delivery_performance = con.execute("""
    WITH line_receipts AS (
        SELECT
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date,

            COALESCE(SUM(pr.Accepted_Quantity), 0)
                AS Total_Accepted_Quantity,

            MAX(
                CASE
                    WHEN pr.Accepted_Quantity > 0
                    THEN pr.Receipt_Date
                END
            ) AS Last_Accepted_Receipt_Date

        FROM Purchase_Order_Lines pol
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        LEFT JOIN Purchase_Receipts pr
            ON pol.PO_Line_ID = pr.PO_Line_ID
        WHERE po.PO_Status = 'Closed'
        GROUP BY
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date
    ),
    completed_lines AS (
        SELECT *,
            CASE
                WHEN Last_Accepted_Receipt_Date
                     <= Expected_Delivery_Date
                THEN 1 ELSE 0
            END AS On_Time_Flag
        FROM line_receipts
        WHERE Total_Accepted_Quantity >= Ordered_Quantity
    )

    SELECT
        s.Supplier_ID,
        s.Supplier_Name,
        COUNT(*) AS Completed_PO_Lines,

        SUM(On_Time_Flag) AS On_Time_Lines,

        COUNT(*) - SUM(On_Time_Flag) AS Late_Lines,

        ROUND(
            SUM(On_Time_Flag) * 100.0 / COUNT(*),
            2
        ) AS On_Time_Delivery_Rate

    FROM completed_lines cl
    JOIN Suppliers s
        ON cl.Supplier_ID = s.Supplier_ID
    GROUP BY s.Supplier_ID, s.Supplier_Name
    ORDER BY On_Time_Delivery_Rate ASC
""").df()

display(supplier_delivery_performance)

In [ ]:
delivery_reconciliation = con.execute("""
    WITH line_receipts AS (
        SELECT
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date,

            COALESCE(SUM(pr.Accepted_Quantity), 0)
                AS Total_Accepted_Quantity,

            MAX(
                CASE
                    WHEN pr.Accepted_Quantity > 0
                    THEN pr.Receipt_Date
                END
            ) AS Last_Accepted_Receipt_Date

        FROM Purchase_Order_Lines pol
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        LEFT JOIN Purchase_Receipts pr
            ON pol.PO_Line_ID = pr.PO_Line_ID
        WHERE po.PO_Status = 'Closed'
        GROUP BY
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date
    ),
    completed_lines AS (
        SELECT *,
            CASE
                WHEN Last_Accepted_Receipt_Date
                     <= Expected_Delivery_Date
                THEN 'On Time'
                ELSE 'Late'
            END AS Delivery_Status
        FROM line_receipts
        WHERE Total_Accepted_Quantity >= Ordered_Quantity
    )

    SELECT
        Delivery_Status,
        COUNT(*) AS PO_Line_Count
    FROM completed_lines
    GROUP BY Delivery_Status
    ORDER BY Delivery_Status
""").df()

display(delivery_reconciliation)

In [ ]:
print("Supplier-level completed lines:",
      supplier_delivery_performance["Completed_PO_Lines"].sum())

print("Supplier-level on-time lines:",
      supplier_delivery_performance["On_Time_Lines"].sum())

print("Supplier-level late lines:",
      supplier_delivery_performance["Late_Lines"].sum())

In [ ]:
supplier_delivery_delays = con.execute("""
    WITH line_receipts AS (
        SELECT
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date,

            SUM(pr.Accepted_Quantity) AS Total_Accepted_Quantity,

            MAX(pr.Receipt_Date) AS Final_Receipt_Date

        FROM Purchase_Order_Lines pol
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        JOIN Purchase_Receipts pr
            ON pol.PO_Line_ID = pr.PO_Line_ID
        WHERE po.PO_Status = 'Closed'
        GROUP BY
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date
    ),
    completed_lines AS (
        SELECT
            *,
            DATE_DIFF(
                'day',
                Expected_Delivery_Date,
                Final_Receipt_Date
            ) AS Delay_Days
        FROM line_receipts
        WHERE Total_Accepted_Quantity >= Ordered_Quantity
    )

    SELECT
        s.Supplier_ID,
        s.Supplier_Name,
        COUNT(*) AS Completed_PO_Lines,

        ROUND(AVG(Delay_Days), 2) AS Average_Delay_Days,

        MAX(Delay_Days) AS Maximum_Delay_Days,

        SUM(CASE WHEN Delay_Days > 0 THEN 1 ELSE 0 END)
            AS Late_Lines

    FROM completed_lines cl
    JOIN Suppliers s
        ON cl.Supplier_ID = s.Supplier_ID
    GROUP BY s.Supplier_ID, s.Supplier_Name
    ORDER BY Average_Delay_Days DESC
""").df()

display(supplier_delivery_delays)

In [ ]:
supplier_type_delivery = con.execute("""
    WITH line_receipts AS (
        SELECT
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date,

            SUM(pr.Accepted_Quantity) AS Total_Accepted_Quantity,
            MAX(pr.Receipt_Date) AS Final_Receipt_Date

        FROM Purchase_Order_Lines pol
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        JOIN Purchase_Receipts pr
            ON pol.PO_Line_ID = pr.PO_Line_ID
        WHERE po.PO_Status = 'Closed'
        GROUP BY
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date
    ),
    completed_lines AS (
        SELECT
            *,
            DATE_DIFF(
                'day',
                Expected_Delivery_Date,
                Final_Receipt_Date
            ) AS Delay_Days
        FROM line_receipts
        WHERE Total_Accepted_Quantity >= Ordered_Quantity
    )

    SELECT
        s.Supplier_Type,
        COUNT(DISTINCT s.Supplier_ID) AS Suppliers,
        COUNT(*) AS Completed_PO_Lines,

        SUM(
            CASE WHEN Delay_Days <= 0 THEN 1 ELSE 0 END
        ) AS On_Time_Lines,

        SUM(
            CASE WHEN Delay_Days > 0 THEN 1 ELSE 0 END
        ) AS Late_Lines,

        ROUND(
            SUM(CASE WHEN Delay_Days <= 0 THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*),
            2
        ) AS On_Time_Delivery_Rate,

        ROUND(AVG(Delay_Days), 2) AS Average_Delay_Days

    FROM completed_lines cl
    JOIN Suppliers s
        ON cl.Supplier_ID = s.Supplier_ID
    GROUP BY s.Supplier_Type
    ORDER BY On_Time_Delivery_Rate DESC
""").df()

display(supplier_type_delivery)

In [ ]:
supplier_leadtime_performance = con.execute("""
    WITH line_receipts AS (
        SELECT
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Supplier_Product_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date,

            SUM(pr.Accepted_Quantity) AS Total_Accepted_Quantity,
            MAX(pr.Receipt_Date) AS Final_Receipt_Date

        FROM Purchase_Order_Lines pol
        JOIN Purchase_Orders po
            ON pol.PO_ID = po.PO_ID
        JOIN Purchase_Receipts pr
            ON pol.PO_Line_ID = pr.PO_Line_ID
        WHERE po.PO_Status = 'Closed'
        GROUP BY
            pol.PO_Line_ID,
            pol.Supplier_ID,
            pol.Supplier_Product_ID,
            pol.Ordered_Quantity,
            po.Expected_Delivery_Date
    ),
    completed_lines AS (
        SELECT
            *,
            DATE_DIFF(
                'day',
                Expected_Delivery_Date,
                Final_Receipt_Date
            ) AS Delay_Days
        FROM line_receipts
        WHERE Total_Accepted_Quantity >= Ordered_Quantity
    )

    SELECT
        s.Supplier_ID,
        s.Supplier_Name,
        COUNT(*) AS Completed_PO_Lines,

        ROUND(AVG(spm.Lead_Time_Days), 2)
            AS Average_Agreed_Lead_Time,

        ROUND(AVG(cl.Delay_Days), 2)
            AS Average_Delay_Days,

        ROUND(
            AVG(spm.Lead_Time_Days + cl.Delay_Days),
            2
        ) AS Average_Actual_Completion_Days

    FROM completed_lines cl
    JOIN Suppliers s
        ON cl.Supplier_ID = s.Supplier_ID
    JOIN Supplier_Product_Mapping spm
        ON cl.Supplier_Product_ID = spm.Supplier_Product_ID

    GROUP BY s.Supplier_ID, s.Supplier_Name
    ORDER BY Average_Actual_Completion_Days DESC
""").df()

display(supplier_leadtime_performance)

In [ ]:
supplier_quality_summary = con.execute("""
    SELECT
        COUNT(DISTINCT PO_Line_ID) AS PO_Lines_With_Receipts,

        SUM(Received_Quantity) AS Total_Received_Units,

        SUM(Accepted_Quantity) AS Total_Accepted_Units,

        SUM(Rejected_Quantity) AS Total_Rejected_Units,

        ROUND(
            SUM(Rejected_Quantity) * 100.0
            / NULLIF(SUM(Received_Quantity), 0),
            2
        ) AS Rejection_Rate,

        COUNT(*) AS Total_Receipt_Records,

        SUM(
            CASE
                WHEN Receipt_Status = 'Partially Rejected'
                THEN 1 ELSE 0
            END
        ) AS Receipts_With_Rejections

    FROM Purchase_Receipts
""").df()

display(supplier_quality_summary)


In [ ]:
supplier_rejection_performance = con.execute("""
    SELECT
        s.Supplier_ID,
        s.Supplier_Name,

        COUNT(DISTINCT pol.PO_Line_ID)
            AS PO_Lines_With_Receipts,

        SUM(pr.Received_Quantity)
            AS Total_Received_Units,

        SUM(pr.Accepted_Quantity)
            AS Accepted_Units,

        SUM(pr.Rejected_Quantity)
            AS Rejected_Units,

        ROUND(
            SUM(pr.Rejected_Quantity) * 100.0
            / NULLIF(SUM(pr.Received_Quantity), 0),
            3
        ) AS Rejection_Rate,

        COUNT(
            DISTINCT CASE
                WHEN pr.Rejected_Quantity > 0
                THEN pol.PO_Line_ID
            END
        ) AS PO_Lines_With_Rejections

    FROM Purchase_Receipts pr
    JOIN Purchase_Order_Lines pol
        ON pr.PO_Line_ID = pol.PO_Line_ID
    JOIN Suppliers s
        ON pol.Supplier_ID = s.Supplier_ID

    GROUP BY
        s.Supplier_ID,
        s.Supplier_Name

    ORDER BY Rejection_Rate DESC
""").df()

display(supplier_rejection_performance)

In [ ]:
overdue_age_df = con.execute("""
SELECT
    CASE
        WHEN DATE_DIFF('day', Expected_Delivery_Date, DATE '2026-10-31') BETWEEN 1 AND 7
            THEN '1–7 Days'
        WHEN DATE_DIFF('day', Expected_Delivery_Date, DATE '2026-10-31') BETWEEN 8 AND 15
            THEN '8–15 Days'
        WHEN DATE_DIFF('day', Expected_Delivery_Date, DATE '2026-10-31') BETWEEN 16 AND 30
            THEN '16–30 Days'
        WHEN DATE_DIFF('day', Expected_Delivery_Date, DATE '2026-10-31') BETWEEN 31 AND 60
            THEN '31–60 Days'
        ELSE 'Over 60 Days'
    END AS Overdue_Age_Bucket,

    COUNT(*) AS Overdue_PO_Count,

    ROUND(
        AVG(DATE_DIFF('day', Expected_Delivery_Date, DATE '2026-10-31')),
        1
    ) AS Average_Days_Overdue

FROM Purchase_Orders
WHERE PO_Status = 'Overdue'
GROUP BY Overdue_Age_Bucket
ORDER BY
    CASE Overdue_Age_Bucket
        WHEN '1–7 Days' THEN 1
        WHEN '8–15 Days' THEN 2
        WHEN '16–30 Days' THEN 3
        WHEN '31–60 Days' THEN 4
        ELSE 5
    END
""").df()

display(overdue_age_df)

In [ ]:
overdue_outstanding_df = con.execute("""
WITH accepted_by_line AS (
    SELECT
        PO_Line_ID,
        SUM(Accepted_Quantity) AS Accepted_Quantity
    FROM Purchase_Receipts
    GROUP BY PO_Line_ID
),
line_outstanding AS (
    SELECT
        pol.PO_ID,
        pol.PO_Line_ID,
        pol.Ordered_Quantity,
        COALESCE(r.Accepted_Quantity, 0) AS Accepted_Quantity,
        GREATEST(
            pol.Ordered_Quantity - COALESCE(r.Accepted_Quantity, 0),
            0
        ) AS Outstanding_Quantity
    FROM Purchase_Order_Lines pol
    LEFT JOIN accepted_by_line r
        ON pol.PO_Line_ID = r.PO_Line_ID
)
SELECT
    COUNT(DISTINCT po.PO_ID) AS Overdue_PO_Count,
    COUNT(l.PO_Line_ID) AS Overdue_PO_Lines,
    SUM(l.Ordered_Quantity) AS Total_Ordered_Units,
    SUM(l.Accepted_Quantity) AS Total_Accepted_Units,
    SUM(l.Outstanding_Quantity) AS Total_Outstanding_Units
FROM Purchase_Orders po
JOIN line_outstanding l
    ON po.PO_ID = l.PO_ID
WHERE po.PO_Status = 'Overdue'
""").df()

display(overdue_outstanding_df)

In [ ]:
overdue_supplier_df = con.execute("""
WITH accepted_by_line AS (
    SELECT
        PO_Line_ID,
        SUM(Accepted_Quantity) AS Accepted_Quantity
    FROM Purchase_Receipts
    GROUP BY PO_Line_ID
),
line_outstanding AS (
    SELECT
        pol.PO_ID,
        pol.PO_Line_ID,
        pol.Ordered_Quantity,
        COALESCE(r.Accepted_Quantity, 0) AS Accepted_Quantity,
        GREATEST(
            pol.Ordered_Quantity - COALESCE(r.Accepted_Quantity, 0),
            0
        ) AS Outstanding_Quantity
    FROM Purchase_Order_Lines pol
    LEFT JOIN accepted_by_line r
        ON pol.PO_Line_ID = r.PO_Line_ID
)
SELECT
    po.Supplier_ID,
    s.Supplier_Name,
    COUNT(DISTINCT po.PO_ID) AS Overdue_PO_Count,
    SUM(l.Outstanding_Quantity) AS Outstanding_Units,
    ROUND(
        100.0 * SUM(l.Outstanding_Quantity)
        / SUM(SUM(l.Outstanding_Quantity)) OVER (),
        2
    ) AS Share_of_Total_Outstanding_Pct
FROM Purchase_Orders po
JOIN line_outstanding l
    ON po.PO_ID = l.PO_ID
JOIN Suppliers s
    ON po.Supplier_ID = s.Supplier_ID
WHERE po.PO_Status = 'Overdue'
GROUP BY po.Supplier_ID, s.Supplier_Name
ORDER BY Outstanding_Units DESC
""").df()

display(overdue_supplier_df)

In [ ]:
overdue_category_df = con.execute("""
WITH accepted_by_line AS (
    SELECT
        PO_Line_ID,
        SUM(Accepted_Quantity) AS Accepted_Quantity
    FROM Purchase_Receipts
    GROUP BY PO_Line_ID
),
line_outstanding AS (
    SELECT
        pol.PO_ID,
        pol.Product_ID,
        GREATEST(
            pol.Ordered_Quantity - COALESCE(r.Accepted_Quantity, 0),
            0
        ) AS Outstanding_Quantity
    FROM Purchase_Order_Lines pol
    LEFT JOIN accepted_by_line r
        ON pol.PO_Line_ID = r.PO_Line_ID
)
SELECT
    p.Category,
    COUNT(DISTINCT l.Product_ID) AS Products_Affected,
    COUNT(DISTINCT po.PO_ID) AS Overdue_PO_Count,
    SUM(l.Outstanding_Quantity) AS Outstanding_Units,
    ROUND(
        100.0 * SUM(l.Outstanding_Quantity)
        / SUM(SUM(l.Outstanding_Quantity)) OVER (),
        2
    ) AS Share_of_Total_Outstanding_Pct
FROM Purchase_Orders po
JOIN line_outstanding l
    ON po.PO_ID = l.PO_ID
JOIN Products p
    ON l.Product_ID = p.Product_ID
WHERE po.PO_Status = 'Overdue'
GROUP BY p.Category
ORDER BY Outstanding_Units DESC
""").df()

display(overdue_category_df)

In [ ]:
overdue_product_df = con.execute("""
WITH accepted_by_line AS (
    SELECT
        PO_Line_ID,
        SUM(Accepted_Quantity) AS Accepted_Quantity
    FROM Purchase_Receipts
    GROUP BY PO_Line_ID
),
line_outstanding AS (
    SELECT
        pol.PO_ID,
        pol.Product_ID,
        GREATEST(
            pol.Ordered_Quantity - COALESCE(r.Accepted_Quantity, 0),
            0
        ) AS Outstanding_Quantity
    FROM Purchase_Order_Lines pol
    LEFT JOIN accepted_by_line r
        ON pol.PO_Line_ID = r.PO_Line_ID
)
SELECT
    l.Product_ID,
    p.Product_Name,
    p.Category,
    COUNT(DISTINCT po.PO_ID) AS Overdue_PO_Count,
    SUM(l.Outstanding_Quantity) AS Outstanding_Units
FROM Purchase_Orders po
JOIN line_outstanding l
    ON po.PO_ID = l.PO_ID
JOIN Products p
    ON l.Product_ID = p.Product_ID
WHERE po.PO_Status = 'Overdue'
GROUP BY l.Product_ID, p.Product_Name, p.Category
HAVING SUM(l.Outstanding_Quantity) > 0
ORDER BY Outstanding_Units DESC
LIMIT 20
""").df()

display(overdue_product_df)

In [ ]:
overdue_po_detail_df = con.execute("""
WITH accepted_by_line AS (
    SELECT
        PO_Line_ID,
        SUM(Accepted_Quantity) AS Accepted_Quantity
    FROM Purchase_Receipts
    GROUP BY PO_Line_ID
),
line_outstanding AS (
    SELECT
        pol.PO_ID,
        pol.PO_Line_ID,
        GREATEST(
            pol.Ordered_Quantity - COALESCE(r.Accepted_Quantity, 0),
            0
        ) AS Outstanding_Quantity
    FROM Purchase_Order_Lines pol
    LEFT JOIN accepted_by_line r
        ON pol.PO_Line_ID = r.PO_Line_ID
),
po_summary AS (
    SELECT
        PO_ID,
        SUM(Outstanding_Quantity) AS Outstanding_Units
    FROM line_outstanding
    GROUP BY PO_ID
)
SELECT
    po.PO_ID,
    po.Supplier_ID,
    s.Supplier_Name,
    po.PO_Date,
    po.Expected_Delivery_Date,
    DATE_DIFF(
        'day', po.Expected_Delivery_Date, DATE '2026-10-31'
    ) AS Days_Overdue,
    ps.Outstanding_Units
FROM Purchase_Orders po
JOIN po_summary ps ON po.PO_ID = ps.PO_ID
JOIN Suppliers s ON po.Supplier_ID = s.Supplier_ID
WHERE po.PO_Status = 'Overdue'
  AND ps.Outstanding_Units > 0
ORDER BY Days_Overdue DESC, Outstanding_Units DESC
LIMIT 20
""").df()

display(overdue_po_detail_df)

In [ ]:
overdue_largest_po_df = con.execute("""
WITH accepted_by_line AS (
    SELECT
        PO_Line_ID,
        SUM(Accepted_Quantity) AS Accepted_Quantity
    FROM Purchase_Receipts
    GROUP BY PO_Line_ID
),
line_outstanding AS (
    SELECT
        pol.PO_ID,
        GREATEST(
            pol.Ordered_Quantity - COALESCE(r.Accepted_Quantity, 0),
            0
        ) AS Outstanding_Quantity
    FROM Purchase_Order_Lines pol
    LEFT JOIN accepted_by_line r
        ON pol.PO_Line_ID = r.PO_Line_ID
),
po_summary AS (
    SELECT
        PO_ID,
        SUM(Outstanding_Quantity) AS Outstanding_Units
    FROM line_outstanding
    GROUP BY PO_ID
)
SELECT
    po.PO_ID,
    po.Supplier_ID,
    s.Supplier_Name,
    po.Expected_Delivery_Date,
    DATE_DIFF(
        'day', po.Expected_Delivery_Date, DATE '2026-10-31'
    ) AS Days_Overdue,
    ps.Outstanding_Units
FROM Purchase_Orders po
JOIN po_summary ps ON po.PO_ID = ps.PO_ID
JOIN Suppliers s ON po.Supplier_ID = s.Supplier_ID
WHERE po.PO_Status = 'Overdue'
  AND ps.Outstanding_Units > 0
ORDER BY Outstanding_Units DESC, Days_Overdue DESC
LIMIT 20
""").df()

display(overdue_largest_po_df)

In [ ]:
price_comparison_df = con.execute("""
SELECT
    p.Product_ID,
    p.Product_Name,
    p.Category,
    sp.Supplier_ID,
    s.Supplier_Name,
    sp.Is_Preferred_Supplier,
    sp.MOQ_Units,
    sp.Unit_Purchase_Price,
    sp.Lead_Time_Days
FROM Supplier_Product_Mapping sp
JOIN Products p
    ON sp.Product_ID = p.Product_ID
JOIN Suppliers s
    ON sp.Supplier_ID = s.Supplier_ID
ORDER BY
    p.Product_ID,
    sp.Is_Preferred_Supplier DESC,
    sp.Unit_Purchase_Price
""").df()

display(price_comparison_df)

In [ ]:
price_comparison_summary_df = con.execute("""
WITH supplier_prices AS (
    SELECT
        sp.Product_ID,
        p.Product_Name,
        p.Category,
        sp.Supplier_ID,
        s.Supplier_Name,
        sp.Is_Preferred_Supplier,
        sp.MOQ_Units,
        sp.Unit_Purchase_Price,
        sp.Lead_Time_Days
    FROM Supplier_Product_Mapping sp
    JOIN Products p
        ON sp.Product_ID = p.Product_ID
    JOIN Suppliers s
        ON sp.Supplier_ID = s.Supplier_ID
),
preferred_prices AS (
    SELECT *
    FROM supplier_prices
    WHERE Is_Preferred_Supplier = TRUE
),
cheapest_alternatives AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY Product_ID
            ORDER BY Unit_Purchase_Price
        ) AS price_rank
    FROM supplier_prices
    WHERE Is_Preferred_Supplier = FALSE
)
SELECT
    pp.Product_ID,
    pp.Product_Name,
    pp.Category,
    pp.Supplier_ID AS Preferred_Supplier_ID,
    pp.Supplier_Name AS Preferred_Supplier,
    pp.MOQ_Units AS Preferred_MOQ,
    pp.Unit_Purchase_Price AS Preferred_Unit_Price,
    pp.Lead_Time_Days AS Preferred_Lead_Time,
    ca.Supplier_ID AS Cheapest_Alternative_ID,
    ca.Supplier_Name AS Cheapest_Alternative,
    ca.MOQ_Units AS Alternative_MOQ,
    ca.Unit_Purchase_Price AS Alternative_Unit_Price,
    ca.Lead_Time_Days AS Alternative_Lead_Time,
    ROUND(
        pp.Unit_Purchase_Price - ca.Unit_Purchase_Price, 2
    ) AS Unit_Price_Difference,
    ROUND(
        100.0 * (pp.Unit_Purchase_Price - ca.Unit_Purchase_Price)
        / pp.Unit_Purchase_Price, 2
    ) AS Potential_Saving_Pct
FROM preferred_prices pp
LEFT JOIN cheapest_alternatives ca
    ON pp.Product_ID = ca.Product_ID
    AND ca.price_rank = 1
ORDER BY Potential_Saving_Pct DESC NULLS LAST
""").df()

display(price_comparison_summary_df)

In [ ]:
price_saving_summary_df = con.execute("""
SELECT
    COUNT(*) AS Total_Products,
    COUNT(CASE
        WHEN Cheapest_Alternative_ID IS NOT NULL THEN 1
    END) AS Products_With_Alternatives,
    COUNT(CASE
        WHEN Potential_Saving_Pct > 0 THEN 1
    END) AS Products_With_Cheaper_Alternatives,
    COUNT(CASE
        WHEN Potential_Saving_Pct <= 0 THEN 1
    END) AS Products_Without_Cheaper_Alternatives,
    ROUND(AVG(Potential_Saving_Pct), 2)
        AS Average_Potential_Saving_Pct,
    ROUND(MAX(Potential_Saving_Pct), 2)
        AS Highest_Potential_Saving_Pct,
    ROUND(MIN(Potential_Saving_Pct), 2)
        AS Lowest_Potential_Saving_Pct
FROM price_comparison_summary_df
""").df()

display(price_saving_summary_df)

In [ ]:
top_price_saving_products_df = price_comparison_summary_df[
    price_comparison_summary_df["Potential_Saving_Pct"] > 0
].sort_values(
    "Potential_Saving_Pct",
    ascending=False
).head(20)

display(top_price_saving_products_df)

In [ ]:
supplier_performance_df = con.execute("""
WITH line_receipts AS (
    SELECT
        pol.PO_Line_ID,
        pol.Supplier_ID,
        po.Expected_Delivery_Date,
        MAX(
            CASE
                WHEN pr.Accepted_Quantity > 0
                THEN pr.Receipt_Date
            END
        ) AS Final_Accepted_Date,
        SUM(pr.Accepted_Quantity) AS Total_Accepted,
        SUM(pr.Rejected_Quantity) AS Total_Rejected,
        SUM(pr.Received_Quantity) AS Total_Received
    FROM Purchase_Order_Lines pol
    JOIN Purchase_Orders po
        ON pol.PO_ID = po.PO_ID
    JOIN Purchase_Receipts pr
        ON pol.PO_Line_ID = pr.PO_Line_ID
    WHERE po.PO_Status = 'Closed'
    GROUP BY
        pol.PO_Line_ID,
        pol.Supplier_ID,
        po.Expected_Delivery_Date
),
supplier_performance AS (
    SELECT
        Supplier_ID,
        COUNT(*) AS Completed_PO_Lines,
        COUNT(*) FILTER (
            WHERE Final_Accepted_Date <= Expected_Delivery_Date
        ) AS On_Time_Lines,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE Final_Accepted_Date <= Expected_Delivery_Date
            ) / COUNT(*), 2
        ) AS On_Time_Rate_Pct,
        ROUND(
            AVG(
                CASE
                    WHEN Final_Accepted_Date > Expected_Delivery_Date
                    THEN DATE_DIFF(
                        'day',
                        Expected_Delivery_Date,
                        Final_Accepted_Date
                    )
                END
            ), 2
        ) AS Average_Days_Late,
        SUM(Total_Received) AS Total_Received_Units,
        SUM(Total_Rejected) AS Total_Rejected_Units,
        ROUND(
            100.0 * SUM(Total_Rejected)
            / NULLIF(SUM(Total_Received), 0), 3
        ) AS Rejection_Rate_Pct
    FROM line_receipts
    WHERE Total_Accepted > 0
    GROUP BY Supplier_ID
),
selected_suppliers AS (
    SELECT
        Product_ID,
        Preferred_Supplier_ID AS Supplier_ID,
        'Preferred' AS Supplier_Role
    FROM price_comparison_summary_df
    WHERE Potential_Saving_Pct > 0

    UNION ALL

    SELECT
        Product_ID,
        Cheapest_Alternative_ID AS Supplier_ID,
        'Cheapest Alternative' AS Supplier_Role
    FROM price_comparison_summary_df
    WHERE Potential_Saving_Pct > 0
)
SELECT
    ps.Product_ID,
    ps.Product_Name,
    ps.Potential_Saving_Pct,
    ss.Supplier_Role,
    ss.Supplier_ID,
    s.Supplier_Name,
    sp.Completed_PO_Lines,
    sp.On_Time_Rate_Pct,
    sp.Average_Days_Late,
    sp.Rejection_Rate_Pct
FROM selected_suppliers ss
JOIN price_comparison_summary_df ps
    ON ss.Product_ID = ps.Product_ID
JOIN Suppliers s
    ON ss.Supplier_ID = s.Supplier_ID
LEFT JOIN supplier_performance sp
    ON ss.Supplier_ID = sp.Supplier_ID
WHERE ps.Potential_Saving_Pct > 0
ORDER BY
    ps.Potential_Saving_Pct DESC,
    ss.Supplier_Role
""").df()

display(
    supplier_performance_df[
        supplier_performance_df["Product_ID"].isin(
            top_price_saving_products_df["Product_ID"]
        )
    ]
)

In [ ]:
supplier_opportunity_shortlist_df = price_comparison_summary_df[
    (price_comparison_summary_df["Potential_Saving_Pct"] >= 10) &
    (price_comparison_summary_df["Alternative_MOQ"] <=
     price_comparison_summary_df["Preferred_MOQ"]) &
    (price_comparison_summary_df["Alternative_Lead_Time"] <=
     price_comparison_summary_df["Preferred_Lead_Time"] + 7)
].sort_values(
    "Potential_Saving_Pct",
    ascending=False
)

display(supplier_opportunity_shortlist_df)

In [ ]:
shortlist_performance_df = supplier_performance_df[
    supplier_performance_df["Product_ID"].isin(
        supplier_opportunity_shortlist_df["Product_ID"]
    )
].copy()

display(
    shortlist_performance_df.sort_values(
        ["Product_ID", "Supplier_Role"]
    )
)

In [ ]:
supplier_pair_comparison_df = shortlist_performance_df.pivot_table(
    index=["Product_ID", "Product_Name", "Potential_Saving_Pct"],
    columns="Supplier_Role",
    values=[
        "On_Time_Rate_Pct",
        "Average_Days_Late",
        "Rejection_Rate_Pct",
        "Completed_PO_Lines"
    ],
    aggfunc="first"
).reset_index()

supplier_pair_comparison_df.columns = [
    "_".join(str(x) for x in col if x != "")
    if isinstance(col, tuple) else col
    for col in supplier_pair_comparison_df.columns
]

supplier_pair_comparison_df["Alternative_Better_On_Time"] = (
    supplier_pair_comparison_df["On_Time_Rate_Pct_Cheapest Alternative"] >
    supplier_pair_comparison_df["On_Time_Rate_Pct_Preferred"]
)

supplier_pair_comparison_df["Alternative_Lower_Average_Delay"] = (
    supplier_pair_comparison_df["Average_Days_Late_Cheapest Alternative"] <
    supplier_pair_comparison_df["Average_Days_Late_Preferred"]
)

supplier_pair_comparison_df["Alternative_Lower_Rejection"] = (
    supplier_pair_comparison_df["Rejection_Rate_Pct_Cheapest Alternative"] <
    supplier_pair_comparison_df["Rejection_Rate_Pct_Preferred"]
)

supplier_pair_summary_df = pd.DataFrame({
    "Comparison": [
        "Alternative has higher on-time rate",
        "Alternative has lower average days late",
        "Alternative has lower rejection rate"
    ],
    "Products": [
        supplier_pair_comparison_df["Alternative_Better_On_Time"].sum(),
        supplier_pair_comparison_df["Alternative_Lower_Average_Delay"].sum(),
        supplier_pair_comparison_df["Alternative_Lower_Rejection"].sum()
    ],
    "Total_Products": [
        len(supplier_pair_comparison_df)
    ] * 3
})

supplier_pair_summary_df["Share_Pct"] = (
    supplier_pair_summary_df["Products"] /
    supplier_pair_summary_df["Total_Products"] * 100
).round(2)

display(supplier_pair_summary_df)

In [ ]:
strong_supplier_opportunities_df = supplier_pair_comparison_df[
    (
        supplier_pair_comparison_df["Potential_Saving_Pct"] >= 10
    ) &
    (
        (
            supplier_pair_comparison_df[
                "On_Time_Rate_Pct_Cheapest Alternative"
            ] >
            supplier_pair_comparison_df[
                "On_Time_Rate_Pct_Preferred"
            ]
        )
        |
        (
            supplier_pair_comparison_df[
                "Average_Days_Late_Cheapest Alternative"
            ] <
            supplier_pair_comparison_df[
                "Average_Days_Late_Preferred"
            ]
        )
    )
].merge(
    supplier_opportunity_shortlist_df[
        [
            "Product_ID",
            "Preferred_Supplier",
            "Cheapest_Alternative",
            "Preferred_MOQ",
            "Alternative_MOQ",
            "Preferred_Lead_Time",
            "Alternative_Lead_Time",
            "Unit_Price_Difference"
        ]
    ],
    on="Product_ID",
    how="left"
).sort_values(
    "Potential_Saving_Pct",
    ascending=False
)

display(strong_supplier_opportunities_df)

In [ ]:
strong_opportunity_summary_df = pd.DataFrame({
    "Metric": [
        "Products shortlisted",
        "Average potential price saving (%)",
        "Average MOQ reduction (%)",
        "Average lead-time change (days)",
        "Alternatives with better on-time rate",
        "Alternatives with lower average days late",
        "Alternatives better on both delivery measures"
    ],
    "Value": [
        len(strong_supplier_opportunities_df),
        strong_supplier_opportunities_df["Potential_Saving_Pct"].mean().round(2),
        (
            (
                strong_supplier_opportunities_df["Preferred_MOQ"] -
                strong_supplier_opportunities_df["Alternative_MOQ"]
            ) /
            strong_supplier_opportunities_df["Preferred_MOQ"] * 100
        ).mean().round(2),
        (
            strong_supplier_opportunities_df["Alternative_Lead_Time"] -
            strong_supplier_opportunities_df["Preferred_Lead_Time"]
        ).mean().round(2),
        strong_supplier_opportunities_df["Alternative_Better_On_Time"].sum(),
        strong_supplier_opportunities_df["Alternative_Lower_Average_Delay"].sum(),
        (
            strong_supplier_opportunities_df["Alternative_Better_On_Time"] &
            strong_supplier_opportunities_df["Alternative_Lower_Average_Delay"]
        ).sum()
    ]
})

display(strong_opportunity_summary_df)

In [ ]:
financial_impact_summary_df = con.execute("""
    SELECT
        COUNT(*) AS Total_Order_Lines,

        SUM(pol.Line_Order_Value) AS Total_Purchasing_Value,

        SUM(
            GREATEST(
                LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                - pol.Required_Quantity,
                0
            ) * pol.Unit_Purchase_Price
        ) AS MOQ_Driven_Over_MOQ_Value,

        SUM(
            GREATEST(
                pol.Ordered_Quantity
                - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                0
            ) * pol.Unit_Purchase_Price
        ) AS Buyer_Driven_Over_MOQ_Value,

        SUM(
            (
                GREATEST(
                    LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                    - pol.Required_Quantity,
                    0
                )
                +
                GREATEST(
                    pol.Ordered_Quantity
                    - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                    0
                )
            ) * pol.Unit_Purchase_Price
        ) AS Total_Over_MOQ_Value,

        ROUND(
            100.0 * SUM(
                (
                    GREATEST(
                        LEAST(pol.Ordered_Quantity, spm.MOQ_Units)
                        - pol.Required_Quantity,
                        0
                    )
                    +
                    GREATEST(
                        pol.Ordered_Quantity
                        - GREATEST(pol.Required_Quantity, spm.MOQ_Units),
                        0
                    )
                ) * pol.Unit_Purchase_Price
            ) / NULLIF(SUM(pol.Line_Order_Value), 0),
            2
        ) AS Over_MOQ_Value_Pct

    FROM Purchase_Order_Lines pol
    JOIN Purchase_Orders po
        ON pol.PO_ID = po.PO_ID
    JOIN Supplier_Product_Mapping spm
        ON pol.Supplier_Product_ID = spm.Supplier_Product_ID

    WHERE po.PO_Status <> 'Cancelled'
""").df()

financial_impact_summary_df

In [ ]:
from pathlib import Path

EXCEL_OUTPUT_DIR = PROJECT_DIR / "data" / "final"
EXCEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_OUTPUT_DIR

In [ ]:
[
    name for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
]

In [ ]:
dataframe_inventory = pd.DataFrame([
    {
        "DataFrame": name,
        "Rows": obj.shape[0],
        "Columns": obj.shape[1]
    }
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
])

dataframe_inventory = dataframe_inventory.sort_values(
    "DataFrame"
).reset_index(drop=True)

display(dataframe_inventory.to_string(index=False))

In [ ]:
key_dataframes = [
    "po_summary",
    "delivery_performance_summary",
    "supplier_quality_summary",
    "overdue_outstanding_df",
    "moq_impact_summary",
    "over_moq_category_summary",
    "over_moq_supplier_type",
    "over_moq_product_summary",
    "supplier_purchase_summary",
    "over_moq_supplier_summary",
    "supplier_delivery_performance",
    "supplier_rejection_performance",
    "strong_supplier_opportunities_df",
    "strong_opportunity_summary_df",
    "price_comparison_summary_df",
    "price_saving_summary_df"
]

for name in key_dataframes:
    obj = globals()[name]
    print(f"\n{name} — {obj.shape}")
    print(obj.columns.tolist())

In [ ]:
# Create clean copies for the final Excel workbook

workbook_data = {
    "PO_Status": po_summary.copy(),
    "Delivery_Summary": delivery_performance_summary.copy(),
    "Quality_Summary": supplier_quality_summary.copy(),
    "Overdue_Summary": overdue_outstanding_df.copy(),
    "Over_MOQ_Summary": financial_impact_summary_df.copy(),
    "Over_MOQ_Category": over_moq_category_summary.copy(),
    "Over_MOQ_Supplier_Type": over_moq_supplier_type.copy(),
    "Over_MOQ_Products": over_moq_product_summary.copy(),
    "Supplier_Opportunities": strong_supplier_opportunities_df.copy(),
    "Opportunity_Summary": strong_opportunity_summary_df.copy(),
    "Price_Comparison": price_comparison_summary_df.copy(),
    "Price_Saving_Summary": price_saving_summary_df.copy()
}

# Rename older terminology in the MOQ impact summary
workbook_data["Over_MOQ_Units"] = moq_impact_summary.rename(columns={
    "MOQ_Driven_Excess_Units": "MOQ_Driven_Over_MOQ_Units",
    "Buyer_Excess_Above_MOQ_Units": "Buyer_Driven_Over_MOQ_Units",
    "Total_Excess_Units": "Total_Over_MOQ_Units"
}).copy()

print("Workbook DataFrames prepared:", len(workbook_data))

In [ ]:
executive_summary_df = pd.DataFrame([
    ["Procurement", "Total Purchase Orders", int(purchase_orders_df.shape[0]), "Orders created during the analysis period"],
    ["Procurement", "Total Order Lines", int(financial_impact_summary_df["Total_Order_Lines"].iloc[0]), "Excludes cancelled purchase orders"],
    ["Procurement", "Total Purchasing Value (INR)", float(financial_impact_summary_df["Total_Purchasing_Value"].iloc[0]), "Excludes cancelled purchase orders"],
    ["Over-MOQ", "MOQ-Driven Over-MOQ Value (INR)", float(financial_impact_summary_df["MOQ_Driven_Over_MOQ_Value"].iloc[0]), "Additional purchase value driven by supplier MOQ"],
    ["Over-MOQ", "Buyer-Driven Over-MOQ Value (INR)", float(financial_impact_summary_df["Buyer_Driven_Over_MOQ_Value"].iloc[0]), "Additional purchase value above both requirement and MOQ"],
    ["Over-MOQ", "Total Over-MOQ Value (INR)", float(financial_impact_summary_df["Total_Over_MOQ_Value"].iloc[0]), "Additional purchase value associated with Over-MOQ quantities"],
    ["Over-MOQ", "Over-MOQ Share of Purchasing Value (%)", float(financial_impact_summary_df["Over_MOQ_Value_Pct"].iloc[0]), "Share of non-cancelled purchasing value"],
    ["Suppliers", "Completed PO Lines", int(delivery_performance_summary["Completed_PO_Lines"].iloc[0]), "Fully completed and accepted order lines"],
    ["Suppliers", "On-Time Delivery Rate (%)", round(100 * delivery_performance_summary["On_Time_Lines"].iloc[0] / delivery_performance_summary["Completed_PO_Lines"].iloc[0], 2), "Based on completed PO lines"],
    ["Suppliers", "Rejected Units", int(supplier_quality_summary["Total_Rejected_Units"].iloc[0]), "Units rejected across recorded receipts"],
    ["Suppliers", "Rejected Quantity Rate (%)", round(100 * supplier_quality_summary["Rejection_Rate"].iloc[0], 2), "Check source field scale before interpreting"],
    ["PO Exceptions", "Overdue Purchase Orders", int(overdue_outstanding_df["Overdue_PO_Count"].iloc[0]), "Incomplete orders overdue at 31 October 2026"],
    ["PO Exceptions", "Outstanding Overdue Units", int(overdue_outstanding_df["Total_Outstanding_Units"].iloc[0]), "Ordered quantity less accepted quantity"],
    ["Supplier Opportunities", "Strong Alternative Supplier Opportunities", int(len(strong_supplier_opportunities_df)), "Shortlisted for commercial and delivery review"]
], columns=["Area", "Metric", "Value", "Explanation"])

display(executive_summary_df)

In [ ]:
supplier_quality_summary[
    [
        "Total_Received_Units",
        "Total_Rejected_Units",
        "Rejection_Rate"
    ]
]

In [ ]:
executive_summary_df.loc[
    executive_summary_df["Metric"] == "Rejected Quantity Rate (%)",
    "Value"
] = supplier_quality_summary["Rejection_Rate"].iloc[0]

# Display financial values in crore for easier reading
executive_summary_df.loc[
    executive_summary_df["Metric"].str.contains("Value \\(INR\\)|Purchasing Value"),
    "Value"
] = (
    executive_summary_df.loc[
        executive_summary_df["Metric"].str.contains("Value \\(INR\\)|Purchasing Value"),
        "Value"
    ] / 10_000_000
)

executive_summary_df.loc[
    executive_summary_df["Metric"].str.contains("Value \\(INR\\)|Purchasing Value"),
    "Metric"
] = executive_summary_df.loc[
    executive_summary_df["Metric"].str.contains("Value \\(INR\\)|Purchasing Value"),
    "Metric"
].str.replace("(INR)", "(₹ crore)", regex=False)

display(executive_summary_df)

In [ ]:
executive_summary_df.loc[
    executive_summary_df["Metric"] == "Over-MOQ Share of Purchasing Value (%)",
    "Value"
] = financial_impact_summary_df["Over_MOQ_Value_Pct"].iloc[0]

display(executive_summary_df)

In [ ]:
# Overall Over-MOQ impact
over_moq_overall_df = financial_impact_summary_df.copy()

# Category-wise impact
over_moq_category_df = over_moq_category_summary.copy()

# Supplier-type impact
over_moq_supplier_type_df = over_moq_supplier_type.copy()

# Product-level impact, sorted by highest financial impact
over_moq_products_df = (
    over_moq_product_summary
    .sort_values(
        "Total_Over_MOQ_Purchase_Value",
        ascending=False
    )
    .reset_index(drop=True)
)

# Add prepared DataFrames to workbook collection
workbook_data.update({
    "Over_MOQ_Overall": over_moq_overall_df,
    "Over_MOQ_Category": over_moq_category_df,
    "Over_MOQ_Supplier_Type": over_moq_supplier_type_df,
    "Over_MOQ_Products": over_moq_products_df
})

print("Over-MOQ analysis tables prepared.")
print("Products covered:", len(over_moq_products_df))


In [ ]:
# Prepare supplier analysis tables
supplier_analysis_data = {
    "Supplier_Purchase": supplier_purchase_summary.copy(),
    "Supplier_Over_MOQ": over_moq_supplier_summary.copy(),
    "Supplier_Delivery": supplier_delivery_performance.copy(),
    "Supplier_Quality": supplier_rejection_performance.copy()
}

# Add them to the workbook collection
workbook_data.update(supplier_analysis_data)

for name, data in supplier_analysis_data.items():
    print(f"{name}: {data.shape[0]} rows, {data.shape[1]} columns")
    print(data.columns.tolist())
    print()

In [ ]:
supplier_analysis_data["Supplier_Purchase"] = (
    supplier_analysis_data["Supplier_Purchase"]
    .sort_values("Purchase_Value", ascending=False)
    .reset_index(drop=True)
)

supplier_analysis_data["Supplier_Over_MOQ"] = (
    supplier_analysis_data["Supplier_Over_MOQ"]
    .sort_values("Total_Over_MOQ_Purchase_Value", ascending=False)
    .reset_index(drop=True)
)

supplier_analysis_data["Supplier_Delivery"] = (
    supplier_analysis_data["Supplier_Delivery"]
    .sort_values("On_Time_Delivery_Rate", ascending=False)
    .reset_index(drop=True)
)

supplier_analysis_data["Supplier_Quality"] = (
    supplier_analysis_data["Supplier_Quality"]
    .sort_values("Rejection_Rate", ascending=True)
    .reset_index(drop=True)
)

print("Supplier analysis tables sorted successfully.")

In [ ]:

# Prepare the 33 strong supplier opportunities
supplier_opportunities_df = (
    strong_supplier_opportunities_df
    .sort_values("Potential_Saving_Pct", ascending=False)
    .reset_index(drop=True)
)

# Prepare opportunity summary
supplier_opportunity_summary_df = strong_opportunity_summary_df.copy()

workbook_data.update({
    "Supplier_Opportunities": supplier_opportunities_df,
    "Opportunity_Summary": supplier_opportunity_summary_df
})

print("Supplier opportunities prepared.")
print("Products shortlisted:", len(supplier_opportunities_df))
print("Columns:", supplier_opportunities_df.columns.tolist())

In [ ]:
# Prepare full product-level price comparison
price_comparison_workbook_df = (
    price_comparison_summary_df
    .sort_values("Potential_Saving_Pct", ascending=False)
    .reset_index(drop=True)
)

# Prepare overall price-saving summary
price_saving_workbook_df = price_saving_summary_df.copy()

workbook_data.update({
    "Price_Comparison": price_comparison_workbook_df,
    "Price_Saving_Summary": price_saving_workbook_df
})

print("Price comparison prepared.")
print("Products covered:", len(price_comparison_workbook_df))
print("Columns:", price_comparison_workbook_df.columns.tolist())

In [ ]:
# Prepare PO exception tables
po_exception_data = {
    "Overdue_Ageing": overdue_age_df.copy(),
    "Overdue_Suppliers": overdue_supplier_df.copy(),
    "Overdue_Categories": overdue_category_df.copy(),
    "Overdue_Products": overdue_product_df.copy(),
    "Oldest_Overdue_POs": overdue_po_detail_df.copy(),
    "Largest_PO_Backlog": overdue_largest_po_df.copy(),
    "Overdue_Summary": overdue_outstanding_df.copy()
}

workbook_data.update(po_exception_data)

for name, data in po_exception_data.items():
    print(f"{name}: {data.shape[0]} rows, {data.shape[1]} columns")

In [ ]:
methodology_notes_df = pd.DataFrame([
    ["Reporting Period", "1 January 2026 to 31 October 2026."],
    ["Reporting Cutoff", "31 October 2026. PO overdue status and overdue days are measured as of this date."],
    ["Cancelled POs", "Cancelled purchase orders are excluded from purchasing value, order-line and Over-MOQ calculations."],
    ["Required Quantity", "The quantity the business genuinely needed before applying supplier MOQ or any additional buyer ordering."],
    ["MOQ-Driven Over-MOQ", "The additional quantity ordered because the supplier's minimum order quantity was higher than the business requirement."],
    ["Buyer-Driven Over-MOQ", "The additional quantity ordered above both the business requirement and the supplier MOQ."],
    ["Total Over-MOQ", "The sum of MOQ-driven and buyer-driven Over-MOQ quantities or their purchase value."],
    ["Over-MOQ Purchase Value", "Calculated using the purchase price of the additional units ordered. It does not represent current unsold inventory or actual working capital tied up."],
    ["Supplier Delivery Performance", "Calculated using completed PO lines. A line is on time when its full accepted quantity was received on or before the expected delivery date."],
    ["Supplier Delivery Limitation", "Supplier delivery performance is based on historical supplier-wide results, not product-specific delivery performance."],
    ["Supplier Lead Time", "PO expected delivery dates were generated independently of supplier-product mapping lead times. Comparisons between mapped lead time and actual completion time are therefore indicative only."],
    ["Rejection Rate", "Rejected units divided by total received units. Rejected units are not counted as accepted fulfilment."],
    ["Overdue Outstanding Quantity", "Ordered quantity less accepted quantity. This can include received-but-rejected units that have not yet been replaced."],
    ["Alternative Supplier Price Saving", "Calculated from the difference between the preferred supplier's unit price and the cheapest alternative supplier's unit price. These are potential, not realised, savings."],
    ["Supplier Opportunity Shortlist", "Initial filter uses at least 10% potential price saving, alternative MOQ no higher than preferred MOQ, and alternative lead time no more than 7 days longer."],
    ["Supplier Opportunity Limitation", "Alternative supplier performance measures are supplier-wide historical figures. Supplier capacity, negotiated terms and product-specific quality should be confirmed before switching."],
    ["Supplier Selection", "The cheapest supplier is not automatically the best choice. MOQ, price, lead time and delivery performance should be considered together."]
], columns=["Topic", "Definition / Assumption"])

workbook_data["Methodology_Notes"] = methodology_notes_df

print("Methodology notes prepared:", len(methodology_notes_df))
display(methodology_notes_df)

In [ ]:
over_moq_units_df = workbook_data["Over_MOQ_Units"]

In [ ]:
final_workbook_sections = {
    "Executive Summary": [
        ("Key Business Metrics", executive_summary_df)
    ],

    "Over-MOQ Analysis": [
        ("Overall Financial Impact", financial_impact_summary_df),
        ("Overall Unit Impact", over_moq_units_df),
        ("Category Breakdown", over_moq_category_df),
        ("Supplier Type Breakdown", over_moq_supplier_type_df)
    ],

    "Product Over-MOQ": [
        ("Product-Level Over-MOQ Impact", over_moq_products_df)
    ],

    "Supplier Analysis": [
        ("Supplier Purchasing", supplier_analysis_data["Supplier_Purchase"]),
        ("Supplier Over-MOQ", supplier_analysis_data["Supplier_Over_MOQ"]),
        ("Supplier Delivery", supplier_analysis_data["Supplier_Delivery"]),
        ("Supplier Quality", supplier_analysis_data["Supplier_Quality"])
    ],

    "Supplier Opportunities": [
        ("Opportunity Summary", supplier_opportunity_summary_df),
        ("Shortlisted Opportunities", supplier_opportunities_df)
    ],

    "Price Comparison": [
        ("Overall Price Saving Summary", price_saving_workbook_df),
        ("Product-Level Price Comparison", price_comparison_workbook_df)
    ],

    "PO Exceptions": [
        ("Overdue Summary", overdue_outstanding_df),
        ("Overdue Ageing", overdue_age_df),
        ("Supplier Breakdown", overdue_supplier_df),
        ("Category Breakdown", overdue_category_df),
        ("Product Breakdown", overdue_product_df),
        ("Oldest Overdue POs", overdue_po_detail_df),
        ("Largest PO Backlog", overdue_largest_po_df)
    ],

    "Methodology Notes": [
        ("Definitions and Assumptions", methodology_notes_df)
    ]
}

for sheet, sections in final_workbook_sections.items():
    print(f"\n{sheet}")
    for section_name, data in sections:
        print(f"  {section_name}: {data.shape[0]} rows × {data.shape[1]} columns")

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

# Output file
output_path = FINAL_DATA_DIR / "ProcureWise_Procurement_Analysis.xlsx"

# Create workbook and write all sections
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    for sheet_name, sections in final_workbook_sections.items():

        current_row = 0

        for section_number, (section_title, df) in enumerate(sections, start=1):

            # Section heading
            heading_row = current_row + 1
            writer.book.create_sheet(sheet_name) if sheet_name not in writer.book.sheetnames else None

            ws = writer.book[sheet_name]
            ws.cell(row=heading_row, column=1, value=section_title)

            # Write data below heading
            df.to_excel(
                writer,
                sheet_name=sheet_name,
                startrow=heading_row,
                index=False
            )

            current_row = heading_row + len(df) + 3

# Apply formatting
wb = load_workbook(output_path)

for ws in wb.worksheets:

    ws.sheet_view.showGridLines = False

    for row in ws.iter_rows():

        for cell in row:

            if cell.value is not None:
                cell.alignment = Alignment(
                    vertical="center",
                    wrap_text=True
                )

    # Identify section headings and table headers
    for row_num in range(1, ws.max_row + 1):

        first_cell = ws.cell(row=row_num, column=1)

        # Section headings
        if first_cell.value is not None and all(
            ws.cell(row=row_num, column=col).value is None
            for col in range(2, ws.max_column + 1)
        ):
            first_cell.font = Font(bold=True, size=12, color="FFFFFF")
            first_cell.fill = PatternFill("solid", fgColor="1F4E78")
            first_cell.alignment = Alignment(vertical="center")
            ws.row_dimensions[row_num].height = 24

        # Table header rows
        elif first_cell.value is not None and row_num > 1:
            previous_row = row_num - 1
            if ws.cell(row=previous_row, column=1).value is not None:
                pass

    # Set readable column widths
    for col_num in range(1, ws.max_column + 1):

        column_letter = get_column_letter(col_num)
        max_length = 0

        for cell in ws[column_letter]:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[column_letter].width = min(max(max_length + 3, 14), 42)

    # Freeze the top row
    ws.freeze_panes = "A2"

wb.save(output_path)

print("Workbook exported successfully.")
print("File:", output_path)
print("Worksheets:", wb.sheetnames)
print("File size (MB):", round(output_path.stat().st_size / (1024 * 1024), 2))

In [ ]:
from openpyxl import load_workbook

wb_check = load_workbook(output_path, data_only=True)

for ws in wb_check.worksheets:
    print(f"\n{'=' * 65}")
    print(f"Sheet: {ws.title}")
    print(f"Dimensions: {ws.max_row} rows × {ws.max_column} columns")

    for row in ws.iter_rows(min_row=1, max_row=min(ws.max_row, 8), values_only=True):
        if any(value is not None for value in row):
            print(row[:4])

    headings = [
        ws.cell(row=r, column=1).value
        for r in range(1, ws.max_row + 1)
        if ws.cell(row=r, column=1).value is not None
        and all(
            ws.cell(row=r, column=c).value is None
            for c in range(2, ws.max_column + 1)
        )
    ]

    print("Section headings:", headings)

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

wb = load_workbook(output_path)

for ws in wb.worksheets:

    ws.sheet_view.showGridLines = False
    current_row = 0

    for section_number, (section_title, df) in enumerate(
        final_workbook_sections[ws.title], start=1
    ):

        heading_row = current_row + 1
        header_row = heading_row + 1
        last_row = header_row + len(df)

        # Format section heading
        heading = ws.cell(row=heading_row, column=1)
        heading.font = Font(bold=True, size=12, color="FFFFFF")
        heading.fill = PatternFill("solid", fgColor="1F4E78")
        heading.alignment = Alignment(vertical="center")
        ws.row_dimensions[heading_row].height = 24

        # Format table headers
        for cell in ws[header_row]:
            if cell.column <= len(df.columns):
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="4472C4")
                cell.alignment = Alignment(
                    horizontal="center",
                    vertical="center",
                    wrap_text=True
                )

        # Add Excel table with filters
        if len(df) > 0:
            table_ref = (
                f"A{header_row}:"
                f"{get_column_letter(len(df.columns))}{last_row}"
            )

            table = Table(
                displayName=f"Table_{ws.title.replace(' ', '').replace('-', '')}_{section_number}",
                ref=table_ref
            )

            style = TableStyleInfo(
                name="TableStyleMedium2",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False
            )

            table.tableStyleInfo = style
            ws.add_table(table)

        # Wrap data cells
        for row in ws.iter_rows(
            min_row=header_row,
            max_row=last_row,
            max_col=len(df.columns)
        ):
            for cell in row:
                cell.alignment = Alignment(
                    vertical="center",
                    wrap_text=True
                )

        current_row = heading_row + len(df) + 3

    # Adjust column widths
    for col_num in range(1, ws.max_column + 1):
        letter = get_column_letter(col_num)
        max_length = max(
            (len(str(cell.value)) for cell in ws[letter]
             if cell.value is not None),
            default=10
        )
        ws.column_dimensions[letter].width = min(
            max(max_length + 3, 14), 42
        )

    ws.freeze_panes = "A3"

wb.save(output_path)

print("Final formatting applied.")
print("Excel tables created:", sum(len(ws.tables) for ws in wb.worksheets))
print("Worksheets:", len(wb.worksheets))
print("File size (MB):", round(output_path.stat().st_size / (1024 * 1024), 2))

In [ ]:
required_tables = [
    "executive_summary_df",
    "financial_impact_summary_df",
    "over_moq_units_df",
    "over_moq_category_df",
    "over_moq_supplier_type_df",
    "over_moq_products_df",
    "supplier_opportunity_summary_df",
    "supplier_opportunities_df",
    "price_saving_workbook_df",
    "price_comparison_workbook_df",
    "methodology_notes_df",
    "overdue_outstanding_df",
    "overdue_age_df",
    "overdue_supplier_df",
    "overdue_category_df",
    "overdue_product_df",
    "overdue_po_detail_df",
    "overdue_largest_po_df"
]

for name in required_tables:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {obj.shape[0]} rows × {obj.shape[1]} columns")
    else:
        print(f"MISSING: {name}")

In [ ]:
po_summary